# Project kernels on IBM QPUs from qBraid — free allowance edition
Prepared 3 October 2026 for Aimé T Shangula. **Start here.** This self-contained notebook uses the exact October kernel engines embedded in your uploaded Qiskit v4 notebook. No source notebooks or datasets are required for the default artificial-data mapping test.

1. In qBraid Lab select a Python 3.11/3.12 Qiskit environment, upload this notebook and run the install cell. Restart the kernel if packages changed.
2. Run all cells with `RUN_HARDWARE=False` first: this runs ideal references and Aer shots, with no account required.
3. Create an IBM Quantum **Open Plan** instance in `us-east`, obtain an API key, and check remaining QPU seconds on the IBM dashboard. Set `RUN_HARDWARE=True` and `AVAILABLE_QPU_SECONDS` to that remaining allowance. Rerun configuration onward. Enter your API key through the hidden prompt or the environment variable `IBM_QUANTUM_API_KEY`.
4. Compilation reports all costs. Submission runs **one selected chunk** and saves its job ID immediately. If queued, rerun only the retrieve cell later. Download the result ZIP before ending your Lab session.

`PROFILE='free'` selects four unweighted project kernels, one signed pair each, three overlap controls and one exact-phase QPE calibration: 8 circuits, 512 shots each. `PROFILE='full'` exposes all 26 latest-family models, three pairs, signed and fidelity measurements, 52 Gram QPE inputs, 6 prime/smooth/random Jacobi QPE inputs and controls. The full suite is chunked; it is **not guaranteed to fit one free allowance**. Full profiles keep exactly the same operators; reduced phase registers reduce resolution. Depth guards may reject dense QPE; they never silently replace the experiment.

The GPU benchmark remains a classical benchmark. Here a classically computed finite Gram matrix is factored into amplitude states. Hardware tests implementation and noise, not efficient feature loading, unseen-input learning, RH, quantum advantage or Majorana validity. No physical QPU results are bundled with this notebook.

In [ ]:
import importlib.metadata as metadata, subprocess, sys
from packaging.requirements import Requirement
requirements=['qiskit==2.5.2','qiskit-aer==0.17.2','qiskit-ibm-runtime==0.50.0',
              'numpy>=1.26,<3','scipy>=1.11,<2','pandas>=2,<4',
              'matplotlib>=3.8,<4','scikit-learn>=1.4,<2','threadpoolctl>=3,<4','mpmath>=1.3,<2','ipython>=8,<10']
needed=[]
for spec in requirements:
    r=Requirement(spec)
    try: installed=metadata.version(r.name)
    except metadata.PackageNotFoundError: installed=None
    if installed is None or installed not in r.specifier: needed.append(spec)
if needed:
    already_loaded=any(x in sys.modules for x in ['numpy','qiskit','qiskit_aer','qiskit_ibm_runtime'])
    subprocess.check_call([sys.executable,'-m','pip','install',*needed])
    if already_loaded: raise RuntimeError('Packages changed. Restart the kernel, then run all cells.')
print('Dependencies ready.')

In [ ]:
from pathlib import Path
import os,sys,json,hashlib,math,time,io,shutil,platform,getpass,importlib
import importlib.metadata as metadata
import numpy as np
import pandas as pd
from scipy.linalg import expm
from IPython.display import display
from qiskit import QuantumCircuit,ClassicalRegister,transpile,qpy
from qiskit.circuit.library import StatePreparation,UnitaryGate,QFTGate
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime.executor_sampler import Sampler

PROFILE='free'             # 'free' or 'full'
RUN_HARDWARE=False         # True enables IBM Open Plan authentication and submission
AVAILABLE_QPU_SECONDS=120  # Replace with actual remaining allowance from IBM dashboard
RUN_BUDGET_SECONDS=120     # Total reserved timeout budget for THIS output folder
MAX_JOB_SECONDS=60         # Enforced QPU-usage timeout, not queue/wall time
SHOTS=512
CHUNK_SIZE=16
CHUNK_INDEX=0              # Full suite: increment after retrieving the preceding chunk
MAX_TWO_QUBIT_GATES=400
MAX_TWO_QUBIT_DEPTH=200
MAX_LOGICAL_QUBITS=7
QPE_PHASE_QUBITS=3         # Full suite project QPE: 3 system + 3 phase qubits
INPUT_BUNDLE=None         # Optional path to exported JSON from the companion notebooks
BACKEND_NAME=None         # None selects an operational least-busy Open Plan QPU
SEED=2026100301
OUT=Path.cwd()/'project_qpu_results'
OUT.mkdir(exist_ok=True)
assert PROFILE in ['free','full'] and 1<=SHOTS<=4096
assert 1<=CHUNK_SIZE<=24 and CHUNK_INDEX>=0
assert 0<MAX_JOB_SECONDS<=RUN_BUDGET_SECONDS<=AVAILABLE_QPU_SECONDS
assert 2<=QPE_PHASE_QUBITS<=4
CONFIG={'RUN_LATEST_FAMILY':True,'FAMILY_ARTIFICIAL_ROWS':32,'FAMILY_SEED':SEED,
        'MAX_TOTAL_QUBITS':14,'SEED':SEED}
ENV={p:metadata.version(p) for p in ['qiskit','qiskit-aer','qiskit-ibm-runtime','numpy','scipy','pandas']}
print('Profile:',PROFILE,'Hardware enabled:',RUN_HARDWARE,'versions:',ENV)

## Exact project definitions and provenance
Definitions are copied byte-for-byte; hashes are checked. Only the kernel-engine correctness checks execute.

In [ ]:
PROJECT_PROVENANCE = {'kernel_family_notebook_sha256': 'ea947bde06ac9bdbb7820c24c7323057e31dd6572fd996c3f742bd4d68fa9918', 'upgraded_v3_notebook_sha256': 'f77b36871bb547f7580594cbc502cd0d72e3a75d628973c7a119489b72c75b81', 'source_engine_sha256': {'fusion_zeta_experiment.py': '14eaea1478534d507db22d773a9e6cb5f41a47d9ebc2d0cc42b2c48406aa71da', 'spectral_scarcity_experiment.py': '39cfd8f6d8914963778bda336fe4e948e275d4dd8af55d1a0eaaaff016b42864', 'separation_navigation_experiment.py': '2df90a0a84d04c2e0817ebd1d9849157c55ba869d1a14063b83d234cd2acd614', 'kernel_family_benchmark.py': 'aec035410de4f45b0872815b981baf91b96ad22708140e9e8d1f726a6d3ec0ae', 'kernel_geometry_checks.py': 'c642d790f91a4e77bf3d54b8cea32321de71260f6b17d020d868e76b19608301'}, 'engine_extraction': 'AST literal ENGINE_SOURCES from supplied notebook; no edits', 'data_type': 'artificial standardized Gaussian vectors; no prediction or physical measurements'}
LATEST_ENGINE_SOURCES = {'fusion_zeta_experiment.py': '"""Documented D-T heat-balance model and an isolated zeta-feature experiment.\n\nRun: python fusion_zeta_experiment.py --out fusion_zeta_results\nDependencies: numpy scipy pandas matplotlib. No downloaded data or GPU required.\nThis is a spatially uniform educational model, not a reactor design or net-electric\npower calculation. Zeta features have no assigned particle or energy identity.\n"""\nfrom pathlib import Path\nimport argparse\nimport json\nimport platform\nimport time\nimport numpy as np\nimport pandas as pd\nimport scipy\nfrom scipy.special import bdtrc, loggamma\nfrom scipy.optimize import brentq, minimize_scalar\nfrom scipy.integrate import solve_ivp\nfrom scipy.linalg import solve\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\n\nKEV_J = 1.602176634e-16\nE_ALPHA = 3500 * KEV_J\nE_FUSION = 17600 * KEV_J\nC_BREM = 5.35e-37  # W m^3 keV^-1/2; nonrelativistic hydrogenic approximation\nN_ION = 1e20  # m^-3; n_e=n_i, T_e=T_i\nSOURCES = {\n    \'reactivity\': \'https://ukaea.github.io/PROCESS/physics-models/fusion_reactions/plasma_bosch_hale/\',\n    \'coefficients\': \'https://ukaea.github.io/PROCESS/source/reference/process/models/physics/fusion_reactions/\',\n    \'power_balance\': \'https://ukaea.github.io/PROCESS/physics-models/plasma_power_balance/\',\n    \'bremsstrahlung\': \'https://library.psfc.mit.edu/catalog/online_pubs/MFE_formulary_2014.pdf\',\n    \'zeta\': \'https://dlmf.nist.gov/25.10\',\n    \'lifetime\': \'https://pdg.lbl.gov/2025/reviews/rpp2025-rev-kinematics.pdf\',\n}\n\n\ndef dt_reactivity(T):\n    """Bosch-Hale D-T Maxwellian reactivity, T in keV, output m^3/s.\n\n    Original fit range 0.2-100 keV; no extrapolation is allowed here.\n    Constants cross-checked against UKAEA PROCESS on 2026-10-01.\n    """\n    T = np.asarray(T, dtype=float)\n    if np.any((T < 0.2) | (T > 100)):\n        raise ValueError(\'D-T fit requires 0.2 <= T <= 100 keV\')\n    theta = T / (1 - T * (0.0151361 + T * (0.00460643 - T * 0.00010675))\n                 / (1 + T * (0.0751886 + T * (0.0135 + T * 1.366e-5))))\n    xi = (34.3827**2 / (4 * theta))**(1/3)\n    return 1e-6 * 1.17302e-9 * theta * np.sqrt(xi / (1124656 * T**3)) * np.exp(-3*xi)\n\n\ndef zeta_euler(s, terms=256):\n    """Euler-accelerated Dirichlet eta, divided by 1-2^(1-s).\n\n    eta_N = sum (-1)^k (k+1)^(-s) Pr[Binomial(N,1/2)>k].\n    Derived by exchanging the finite Euler-transformation sums. Suitable for\n    these low-lying roots only; this is not a rigorous root certificate.\n    """\n    k = np.arange(terms)\n    weights = bdtrc(k, terms, 0.5)\n    return np.sum((-1.0)**k * weights * np.exp(-s*np.log(k+1))) / (1-np.exp((1-s)*np.log(2)))\n\n\ndef hardy_z(t, terms=256):\n    theta = np.imag(loggamma(0.25 + 0.5j*t)) - 0.5*t*np.log(np.pi)\n    return float(np.real(np.exp(1j*theta)*zeta_euler(0.5+1j*t, terms)))\n\n\ndef print_zeros(count=12):\n    grid = np.arange(10, 65.01, 0.1)\n    vals = [hardy_z(t) for t in grid]\n    roots = [brentq(hardy_z, a, b, xtol=1e-12)\n             for a, b, fa, fb in zip(grid[:-1], grid[1:], vals[:-1], vals[1:]) if fa*fb < 0]\n    if len(roots) < count:\n        raise RuntimeError(\'Not enough sign-change roots found\')\n    roots = np.array(roots[:count])\n    residual = np.array([abs(zeta_euler(0.5+1j*t, 384)) for t in roots])\n    shift = np.array([abs(brentq(lambda u: hardy_z(u, 192), t-0.02, t+0.02)-t) for t in roots])\n    return pd.DataFrame({\'index\': np.arange(1,count+1), \'ordinate\': roots,\n                         \'residual_384_terms\': residual, \'root_shift_192_terms\': shift})\n\n\ndef ignition_triple(T, deposited=1.0, fraction=0.5):\n    """Minimum n_i T tau_E with no auxiliary heat, pure replenished fuel.\n\n    Denominator is heating-minus-radiation per n^2. tau_E here represents\n    transport loss only; radiation is accounted for separately.\n    """\n    T = np.asarray(T, dtype=float)\n    denominator = deposited * fraction*(1-fraction)*dt_reactivity(T)*E_ALPHA - C_BREM*np.sqrt(T)\n    return np.divide(3*KEV_J*T*T, denominator, out=np.full_like(T,np.inf), where=denominator>0)\n\n\ndef ignition_scan():\n    rows = []\n    for dep in [1.0, 0.9, 0.5, 0.2]:\n        opt = minimize_scalar(lambda T: ignition_triple(T, dep), bounds=(5,50), method=\'bounded\')\n        rows.append({\'alpha_deposited_fraction\': dep, \'optimal_T_keV\': opt.x,\n                     \'minimum_triple_keV_s_m3\': opt.fun,\n                     \'transport_tau_E_s_at_n_1e20\': opt.fun/(N_ION*opt.x),\n                     \'optimal_D_fraction\': 0.5})\n    return pd.DataFrame(rows)\n\n\ndef transient(tau_E, tau_escape, duration=30, initial_T=15):\n    """Two energy reservoirs: thermal U and fast-alpha A, both J/m^3.\n\n    Alpha slowing-down time is a fixed illustrative 0.3 s, not particle decay.\n    Fresh fuel replaces burned fuel, ash is removed; no fueling/ash physics.\n    Six cumulative energy channels are integrated alongside the reservoirs.\n    """\n    tau_s = 0.3\n    heat_capacity = 3*N_ION*KEV_J\n\n    def rhs(t, y):\n        U, A = y[:2]\n        # Solver trial stages may slightly cross a termination boundary. The\n        # terminal event, not clipping the reported solution, limits validity.\n        T = np.clip(U/heat_capacity, 0.2, 100)\n        pf = 0.25*N_ION**2*dt_reactivity(T)*E_FUSION\n        pa = pf*E_ALPHA/E_FUSION\n        pb = C_BREM*N_ION**2*np.sqrt(T)\n        transport, escape = U/tau_E, A/tau_escape\n        aux = 0.0\n        return [aux+A/tau_s-transport-pb, pa-A/tau_s-escape,\n                aux, pa, transport, pb, escape, pf]\n\n    def cold(t, y): return y[0]/heat_capacity-0.2\n    def hot(t, y): return 100-y[0]/heat_capacity\n    cold.terminal = hot.terminal = True\n    cold.direction = hot.direction = -1\n    initial = np.zeros(8)\n    initial[0] = heat_capacity*initial_T\n    sol = solve_ivp(rhs, (0,duration), initial, rtol=1e-9, atol=1e-5,\n                    max_step=0.05, events=[cold,hot], dense_output=True)\n    if not sol.success:\n        raise RuntimeError(sol.message)\n    t = np.linspace(0,sol.t[-1],601)\n    y = sol.sol(t)\n    closure = y[0]+y[1]-initial[0]-y[2]-y[3]+y[4]+y[5]+y[6]\n    scale = max(float(y[3,-1]),initial[0])\n    df = pd.DataFrame({\'time_s\':t,\'T_keV\':y[0]/heat_capacity,\'thermal_J_m3\':y[0],\n                       \'fast_alpha_J_m3\':y[1], \'fusion_J_m3\':y[7],\n                       \'energy_closure_J_m3\':closure})\n    row = {\'tau_E_s\':tau_E,\'alpha_escape_s\':tau_escape,\n           \'deposited_fraction_equilibrium\':tau_escape/(tau_escape+tau_s),\n           \'final_time_s\':sol.t[-1],\'final_T_keV\':df.T_keV.iloc[-1],\n           \'final_fusion_energy_MJ_m3\':y[7,-1]/1e6,\n           \'max_relative_energy_closure\':float(np.max(np.abs(closure))/scale),\n           \'termination\':\'fit_boundary\' if sol.status==1 else \'completed\'}\n    return df,row\n\n\ndef equilibria(tau_E=3.0, tau_escape=3.0):\n    tau_s=0.3; dep=tau_escape/(tau_escape+tau_s); C=3*N_ION*KEV_J\n    pa=lambda T: .25*N_ION**2*dt_reactivity(T)*E_ALPHA\n    F=lambda T: dep*pa(T)-C*T/tau_E-C_BREM*N_ION**2*np.sqrt(T)\n    grid=np.linspace(.2,100,2001); rows=[]\n    for a,b in zip(grid[:-1],grid[1:]):\n        if F(a)*F(b)<0:\n            T=brentq(F,a,b); h=1e-3; derivative=(pa(T+h)-pa(T-h))/(2*h)\n            jac=np.array([[-1/tau_E-C_BREM*N_ION**2/(2*np.sqrt(T)*C),1/tau_s],\n                          [derivative/C,-1/tau_s-1/tau_escape]])\n            ev=np.linalg.eigvals(jac)\n            rows.append({\'T_keV\':T,\'max_real_growth_s_inverse\':float(np.max(ev.real)),\n                         \'thermal_stability\':\'stable\' if np.max(ev.real)<0 else \'unstable\'})\n    return pd.DataFrame(rows)\n\n\ndef features(x, freq, scale):\n    phase=2*np.pi*scale*x[:,None]*freq[None,:]\n    # Positive resolvent-like diagonal damping. This is a feature filter, not\n    # a claimed eigensystem of the Dirichlet operator in the supplied note.\n    damping=1/(1+(scale*freq)**2)\n    return np.column_stack([np.ones(len(x)),np.cos(phase)*damping,np.sin(phase)*damping])\n\n\ndef fit_predict(F, y, G, ridge):\n    penalty=np.eye(F.shape[1]); penalty[0,0]=0\n    coef=solve(F.T@F+len(y)*ridge*penalty,F.T@y,assume_a=\'pos\')\n    return G@coef\n\n\ndef benchmark(zeros, seeds=8):\n    """Approximate log Maxwellian reactivity from synthetic noisy observations.\n\n    Targets: analytic Bosch-Hale curve (not laboratory measurements). Independent\n    train/validation/test points in [2,40] keV, 1% log observation noise in train\n    and validation. Test compares to clean analytic values. No physical coupling\n    is inferred. Zeta/Fourier/scrambled models have the same 25 coefficients.\n    Chebyshev degree is validation-selected from 6/12/18/24 (up to 25 coefficients);\n    RBF kernel uses all training points and is a stronger baseline.\n    Hyperparameters selected on validation only; no post-test retuning.\n    """\n    start=time.perf_counter(); rows=[]; demo={}\n    fz=zeros/zeros[0]; ff=np.linspace(fz.min(),fz.max(),len(fz))\n    ridges=[1e-8,1e-6,1e-4,1e-2]\n    scales=[.25,.5,1.,2.]\n    for seed in range(seeds):\n        rng=np.random.default_rng(8100+seed)\n        train=np.sort(rng.uniform(0,1,120)); valid=np.sort(rng.uniform(0,1,80))\n        test=np.sort(rng.uniform(0,1,400))\n        temperature=lambda x:2*20**x\n        truth=lambda x:np.log(dt_reactivity(temperature(x))/1e-22)\n        yt=truth(train)+rng.normal(0,.01,len(train))\n        yv=truth(valid)+rng.normal(0,.01,len(valid)); ye=truth(test)\n        fs=np.r_[fz[0],fz[0]+np.cumsum(rng.permutation(np.diff(fz)))]\n        methods={\'Zeta features\':fz,\'Uniform-frequency features\':ff,\'Scrambled-gap features\':fs,\n                 \'Chebyshev\':None,\'RBF kernel\':None}\n        for name,freq in methods.items():\n            best=None\n            for scale in scales if freq is not None or name==\'RBF kernel\' else [6,12,18,24]:\n                if freq is not None:\n                    F=features(train,freq,scale); V=features(valid,freq,scale); E=features(test,freq,scale)\n                elif name==\'Chebyshev\':\n                    F=np.polynomial.chebyshev.chebvander(2*train-1,int(scale))\n                    V=np.polynomial.chebyshev.chebvander(2*valid-1,int(scale))\n                    E=np.polynomial.chebyshev.chebvander(2*test-1,int(scale))\n                else:\n                    length=scale*.25\n                    F=np.exp(-.5*((train[:,None]-train[None,:])/length)**2)\n                    V=np.exp(-.5*((valid[:,None]-train[None,:])/length)**2)\n                    E=np.exp(-.5*((test[:,None]-train[None,:])/length)**2)\n                for ridge in ridges:\n                    if name==\'RBF kernel\':\n                        mean=yt.mean(); c=solve(F+len(yt)*ridge*np.eye(len(yt)),yt-mean,assume_a=\'pos\')\n                        predv=V@c+mean; prede=E@c+mean\n                    else:\n                        predv=fit_predict(F,yt,V,ridge); prede=fit_predict(F,yt,E,ridge)\n                    error=float(np.mean((predv-yv)**2))\n                    if best is None or error<best[0]:best=(error,scale,ridge,prede)\n            pred=best[3]; rel=np.expm1(pred-ye)\n            rows.append({\'seed\':seed,\'method\':name,\'validation_log_MSE\':best[0],\n                         \'chosen_scale\':best[1],\'chosen_ridge\':best[2],\n                         \'test_log_RMSE\':float(np.sqrt(np.mean((pred-ye)**2))),\n                         \'test_relative_RMSE_percent\':float(100*np.sqrt(np.mean(rel**2)))})\n            if seed==0:demo[name]=(temperature(test),pred,ye)\n    frame=pd.DataFrame(rows)\n    summary=frame.groupby(\'method\').test_relative_RMSE_percent.agg([\'mean\',\'std\',\'count\'])\n    summary[\'standard_error\']=summary[\'std\']/np.sqrt(summary[\'count\'])\n    summary=summary.sort_values(\'mean\').reset_index()\n    # Reordering complete sin/cos mode pairs does not change the Gram kernel.\n    x=np.linspace(0,1,40); Phi=features(x,fz,.5); K=Phi@Phi.T\n    perm=np.random.default_rng(5).permutation(len(fz)); Psi=features(x,fz[perm],.5)\n    invariance=np.linalg.norm(K-Psi@Psi.T)/np.linalg.norm(K)\n    psd=float(np.linalg.eigvalsh(K).min())\n    checks={\'mode_reordering_relative_Gram_change\':float(invariance),\n            \'minimum_Gram_eigenvalue\':psd,\'benchmark_seconds\':time.perf_counter()-start}\n    return frame,summary,demo,checks\n\n\ndef main(out=\'fusion_zeta_results\'):\n    out=Path(out); out.mkdir(parents=True,exist_ok=True)\n    zeros=print_zeros(); ignition=ignition_scan(); eq=equilibria()\n    runs=[]; transient_rows=[]\n    for tauE,escape,label in [(1.,3.,\'Short heat confinement\'),(3.,3.,\'Longer heat confinement\'),\n                              (3.,.03,\'Fast alpha escape\')]:\n        curve,row=transient(tauE,escape); row[\'case\']=label\n        transient_rows.append(row); runs.append((label,curve))\n        curve.to_csv(out/(label.lower().replace(\' \',\'_\')+\'.csv\'),index=False)\n    trans=pd.DataFrame(transient_rows)\n    frame,summary,demo,checks=benchmark(zeros.ordinate.to_numpy())\n    zeros.to_csv(out/\'zeta_zeros.csv\',index=False)\n    ignition.to_csv(out/\'ignition_optimization.csv\',index=False)\n    eq.to_csv(out/\'thermal_equilibria.csv\',index=False)\n    trans.to_csv(out/\'transient_summary.csv\',index=False)\n    frame.to_csv(out/\'feature_benchmark_all_seeds.csv\',index=False)\n    summary.to_csv(out/\'feature_benchmark_summary.csv\',index=False)\n    assert abs(dt_reactivity(10)/1.1361654705836232e-22-1)<1e-10\n    assert zeros.residual_384_terms.max()<1e-9\n    assert zeros.root_shift_192_terms.max()<1e-8\n    assert abs(zeros.ordinate.iloc[0]-14.134725141734694)<1e-9\n    assert checks[\'mode_reordering_relative_Gram_change\']<1e-12\n    assert checks[\'minimum_Gram_eigenvalue\']>-1e-10\n    assert trans.max_relative_energy_closure.max()<1e-8\n    assert 2.5e21 < ignition.minimum_triple_keV_s_m3.iloc[0] < 3.5e21\n    checks.update({\'zero_max_residual\':float(zeros.residual_384_terms.max()),\n                   \'zero_max_root_shift\':float(zeros.root_shift_192_terms.max()),\n                   \'maximum_relative_energy_closure\':float(trans.max_relative_energy_closure.max()),\n                   \'meaningful_checks_passed\':True})\n    plt.rcParams.update({\'font.size\':10,\'axes.spines.top\':False,\'axes.spines.right\':False})\n    fig,axes=plt.subplots(2,2,figsize=(12,8),layout=\'constrained\')\n    T=np.linspace(5,45,500)\n    for dep in [1.,.9,.5,.2]:axes[0,0].plot(T,ignition_triple(T,dep)/1e21,label=f\'{100*dep:g}% alpha deposition\')\n    axes[0,0].set(xlabel=\'Temperature (keV)\',ylabel=\'Required n T tau_E (10^21 keV s/m^3)\',\n                  title=\'Idealized ignition: retention matters\',ylim=(0,25));axes[0,0].legend(fontsize=8)\n    for label,curve in runs:axes[0,1].plot(curve.time_s,curve.T_keV,label=label)\n    axes[0,1].set(xlabel=\'Time (s)\',ylabel=\'Temperature (keV)\',title=\'Thermal and fast-alpha energy reservoirs\');axes[0,1].legend(fontsize=8)\n    axes[1,0].barh(summary.method,summary[\'mean\'],xerr=summary.standard_error,color=[\'#477a8b\']*len(summary))\n    axes[1,0].set(xlabel=\'Held-out reactivity relative RMSE (%)\',title=\'Synthetic curve approximation: 8 seeds\');axes[1,0].invert_yaxis()\n    for name,(tt,pred,truth) in demo.items():axes[1,1].plot(tt,100*np.expm1(pred-truth),label=name)\n    axes[1,1].set(xlabel=\'Temperature (keV)\',ylabel=\'Prediction error (%)\',title=\'Held-out errors, seed 0\');axes[1,1].legend(fontsize=7)\n    fig.suptitle(\'D-T heat balance and zeta features | educational simulation\',fontsize=16)\n    fig.savefig(out/\'fusion_zeta_summary.png\',dpi=180);plt.close(fig)\n    manifest={\'date\':\'2026-10-01\',\'python\':platform.python_version(),\'numpy\':np.__version__,\n              \'scipy\':scipy.__version__,\'sources\':SOURCES,\'checks\':checks,\n              \'data\':\'synthetic observations of an analytic fit; no device measurements\',\n              \'model\':\'uniform pure equimolar replenished DT; Ti=Te; no ash; constant transport tauE\',\n              \'new_branch\':\'ZetaFrequencyFeatures-Fusion-v1; not frozen project kernels\',\n              \'noise_log_std\':.01,\'seeds\':8,\'zero_modes\':12,\n              \'not_claimed\':[\'particle production\',\'Ricci-flow dynamics\',\'net electric power\',\n                             \'quantum advantage\',\'physical energy extraction from zeta zeros\']}\n    (out/\'manifest.json\').write_text(json.dumps(manifest,indent=2))\n    print(\'IGNITION OPTIMIZATION\\n\'+ignition.to_string(index=False))\n    print(\'\\nTRANSIENTS\\n\'+trans.to_string(index=False))\n    print(\'\\nFEATURE BENCHMARK\\n\'+summary.to_string(index=False))\n    print(\'\\nCHECKS\\n\'+json.dumps(checks,indent=2))\n    return {\'zeros\':zeros,\'ignition\':ignition,\'transients\':trans,\'equilibria\':eq,\n            \'benchmark\':frame,\'summary\':summary,\'checks\':checks,\'out\':out}\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--out\',default=\'fusion_zeta_results\')\n    main(parser.parse_args().out)\n', 'spectral_scarcity_experiment.py': '"""Observation-removal test of frozen kernels and supervised spectral weighting.\n\nRun: OPENBLAS_NUM_THREADS=1 python spectral_scarcity_experiment.py --seeds 12\nCPU only. numpy scipy pandas matplotlib scikit-learn threadpoolctl required.\nNo project files are overwritten. No downloaded datasets or privileged test data.\n"""\nimport os\nos.environ.setdefault(\'OPENBLAS_NUM_THREADS\',\'1\')\nos.environ.setdefault(\'OMP_NUM_THREADS\',\'1\')\nfrom pathlib import Path\nimport argparse\nimport hashlib\nimport json\nimport time\nimport platform\nimport numpy as np\nimport pandas as pd\nimport scipy\nfrom scipy.linalg import eigh\nfrom scipy.stats import t as student_t\nimport sklearn\nfrom sklearn.datasets import load_digits, make_classification\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.svm import SVC\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.ensemble import HistGradientBoostingClassifier\nfrom sklearn.metrics import roc_auc_score, average_precision_score, balanced_accuracy_score\nfrom threadpoolctl import threadpool_limits\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\n\nCONFIG={\n    \'version\':\'SpectralScarcity-v1\', \'date\':\'2026-10-01\', \'seeds\':12,\n    \'budgets\':[32,64,128,256], \'gamma_values\':[.0125,.05,.2],\n    \'original_C_values\':np.logspace(-2,2,9).tolist(),\n    \'weighted_C_values\':[.1,1.,10.], \'weight_strengths\':[0.,.5,1.],\n    \'validation_fraction\':.25, \'candidates_per_method\':27,\n    \'primary_dataset\':\'fusion_hidden_losses\', \'primary_kernel\':\'ProcessUniform\',\n    \'primary_metric\':\'ROC_AUC\',\'scarce_budgets\':[32,64],\'abundant_budget\':256,\n    \'meaningful_AUC_difference\':.01,\n    \'interpretation\':\'Exploratory, conditional on each fixed test set; not population-level validation\',\n}\nPHI=(1+np.sqrt(5))/2\nKERNELS=[\'RBF\',\'ProcessUniform\',\'ProcessLamb-PSD\',\'BosonicX-LogPoly-RG\']\n\n\ndef sqdist(A,B):\n    return np.maximum(np.sum(A*A,axis=1)[:,None]+np.sum(B*B,axis=1)[None,:]-2*A@B.T,0)\n\n\ndef process_transform(X,depth=2,a=.18,c=.12):\n    Z=np.array(X,dtype=float,copy=True)\n    for t in range(depth):Z=(Z+a*np.sin((t+1)*Z))/(1+c*np.cos(Z))\n    return Z\n\n\ndef rg_scale(d):return 1-.32/(np.log(d+2)+np.log(PHI))\n\n\ndef phase_feature(X):\n    Z=process_transform(X);energy=.5*np.mean(Z*Z,axis=1)\n    return rg_scale(X.shape[1])*(energy+.12*np.log1p(np.abs(energy))**2)\n\n\ndef bosonic_warp(X):\n    Z=process_transform(X);scale=rg_scale(X.shape[1])\n    Xw=scale*(Z+.10*np.sin(PHI*Z)/(1+np.abs(Z)))\n    E=.5*np.sum(Xw*Xw,axis=1)+.1*np.sum(np.sin(Xw),axis=1)\n    return Xw,scale*(E+.18*np.log1p(np.abs(E))**2)\n\n\ndef kernel(A,B,name,gamma):\n    """Float64 NumPy translation of supplied notebook cell 4; formulas unchanged."""\n    if name==\'RBF\':return np.exp(-gamma*sqdist(A,B))\n    if name==\'Laplacian\':\n        return np.exp(-gamma*np.sum(np.abs(A[:,None,:]-B[None,:,:]),axis=2))\n    if name==\'ProcessUniform\':\n        d=A.shape[1];base=np.exp(-gamma*sqdist(process_transform(A),process_transform(B)))\n        return d/(d+5)*base+5/(d+5)*(A@B.T/max(d,1))\n    if name==\'ProcessLamb-PSD\':\n        base=np.exp(-gamma*sqdist(process_transform(A),process_transform(B)))\n        pa,pb=1.7*phase_feature(A),1.7*phase_feature(B)\n        return base*(1+.20*np.cos(pa[:,None]-pb[None,:]))/1.20\n    if name==\'BosonicX-LogPoly-RG\':\n        XA,EA=bosonic_warp(A);XB,EB=bosonic_warp(B)\n        occ=(1-.55)/np.maximum(1-.55*np.exp(-gamma*sqdist(XA,XB)),1e-7)\n        return np.exp(-.35*np.abs(EA[:,None]-EB[None,:]))*occ\n    raise ValueError(name)\n\n\ndef center_kernel(K,cross):\n    mean=K.mean(axis=0);grand=K.mean()\n    return K-mean[None,:]-mean[:,None]+grand, cross-cross.mean(axis=1)[:,None]-mean[None,:]+grand\n\n\ndef spectral_map(K,cross,y,mode,strength,seed):\n    """Train-only eigenbasis; consistent extension to genuinely unseen rows.\n\n    Phi_train = U sqrt(lambda); Phi_new = K_new U / sqrt(lambda).\n    Supervised weights are shared across three eigenvalue-order bands, rather\n    than one fitted weight for each training example. They stay nonnegative.\n    """\n    Kc,Cc=center_kernel(K,cross)\n    vals,U=eigh(.5*(Kc+Kc.T),check_finite=False)\n    keep=vals>max(vals.max(),1)*1e-10\n    vals=vals[keep][::-1];U=U[:,keep][:,::-1]\n    if not len(vals):raise ValueError(\'Kernel has no retained positive spectrum\')\n    tr=U*np.sqrt(vals)[None,:];other=(Cc@U)/np.sqrt(vals)[None,:]\n    weight=np.ones(len(vals));band_scores=[]\n    if mode in [\'weighted\',\'sham\']:\n        target=np.asarray(y,dtype=float)\n        if mode==\'sham\':target=np.random.default_rng(seed+90000).permutation(target)\n        target=target-target.mean();projection=U.T@target\n        bands=np.array_split(np.arange(len(vals)),3)\n        for band in bands:\n            score=float(np.sum(vals[band]*projection[band]**2)/(np.sqrt(np.sum(vals[band]**2))+1e-15))\n            band_scores.append(score)\n        profile=.1+.9*np.array(band_scores)/(np.mean(band_scores)+1e-15)\n        for band,w in zip(bands,profile):weight[band]=(1-strength)+strength*w\n    elif mode==\'power\':weight=(vals/max(vals.mean(),1e-15))**(strength-1)\n    elif mode!=\'original\':raise ValueError(mode)\n    tr=tr*np.sqrt(weight)[None,:];other=other*np.sqrt(weight)[None,:]\n    norm=np.sum(tr*tr)/len(tr)\n    tr/=np.sqrt(norm);other/=np.sqrt(norm)\n    scaled=vals*weight;probs=scaled/scaled.sum()\n    diagnostic={\'effective_rank\':float(np.exp(-np.sum(probs*np.log(probs)))),\n                \'retained_modes\':len(vals),\'band_scores\':band_scores,\n                \'minimum_weight\':float(weight.min()),\'maximum_weight\':float(weight.max()),\n                \'train_alignment\':float(np.sum((tr.T@(np.asarray(y)-np.mean(y)))**2)/\n                                         (np.linalg.norm(tr.T@tr)*np.sum((np.asarray(y)-np.mean(y))**2)+1e-15))}\n    return tr@tr.T,other@tr.T,diagnostic\n\n\ndef reactivity(T):\n    th=T/(1-T*(.0151361+T*(.00460643-T*.00010675))/(1+T*(.0751886+T*(.0135+T*1.366e-5))))\n    xi=(34.3827**2/(4*th))**(1/3)\n    return 1e-6*1.17302e-9*th*np.sqrt(xi/(1124656*T**3))*np.exp(-3*xi)\n\n\ndef fusion_dataset(size=1600):\n    """Two-reservoir uniform DT simulator. Rates stay hidden from every learner.\n\n    Class 0: enhanced thermal transport loss. Class 1: enhanced alpha escape.\n    Eight noisy temperature readings and eight noisy log-neutron-rate readings,\n    plus eight nuisance channels. These are synthetic, not reactor measurements.\n    """\n    rng=np.random.default_rng(77001);y=np.tile([0,1],size//2)\n    rng.shuffle(y);n=rng.uniform(.7,1.3,size)*1e20\n    tauE=np.where(y==0,rng.uniform(1.2,2.4,size),rng.uniform(2.4,3.6,size))\n    escape=np.where(y==0,rng.uniform(1,4,size),rng.uniform(.03,.3,size))\n    slow=rng.uniform(.15,.5,size);T0=rng.uniform(12,18,size)\n    C=3*n*1.602176634e-16;U=C*T0;A=np.zeros(size)\n    observed=[];times=np.array([.10,.25,.50,.75,1.,1.5,2.5,4.])\n    step=.01;capture=set(np.rint(times/step).astype(int));terminated=np.zeros(size,bool)\n    def rhs(u,a):\n        T=np.clip(u/C,.2,100);pa=.25*n*n*reactivity(T)*3500*1.602176634e-16\n        return a/slow-u/tauE-5.35e-37*n*n*np.sqrt(T),pa-a/slow-a/escape\n    for j in range(1,401):\n        a,b=rhs(U,A);c,d=rhs(U+step*a/2,A+step*b/2)\n        e,f=rhs(U+step*c/2,A+step*d/2);g,h=rhs(U+step*e,A+step*f)\n        U+=step*(a+2*c+2*e+g)/6;A+=step*(b+2*d+2*f+h)/6\n        terminated|=(U/C<=.2)|(U/C>=100)\n        U=np.maximum(U,.2*C);U=np.minimum(U,100*C)\n        # No trajectory reaches a fit boundary in the delivered parameter range;\n        # boundary checks below fail instead of treating clipping as physics.\n        if j in capture:observed.append((U/C).copy())\n    if terminated.any():raise RuntimeError(\'Simulator fit boundary reached; revise declared domain before testing\')\n    temp=np.column_stack(observed)\n    neutron=.25*n[:,None]**2*reactivity(temp)\n    measuredT=temp+rng.normal(size=temp.shape)*(.25+.03*temp)\n    measuredN=np.log(neutron/1e18)+rng.normal(0,.10,size=neutron.shape)\n    nuisance=rng.normal(size=(size,8))\n    X=np.column_stack([measuredT,measuredN,nuisance])\n    metadata=np.column_stack([n,tauE,escape,slow,T0])\n    return X,y,metadata\n\n\ndef datasets():\n    X,y,meta=fusion_dataset()\n    data={\'fusion_hidden_losses\':(X,y,400,{\'source\':\'synthetic two-reservoir DT model\',\'hidden_parameters\':meta})}\n    X,y=make_classification(n_samples=1600,n_features=24,n_informative=12,n_redundant=4,\n                            weights=[.72,.28],class_sep=1.,flip_y=.03,random_state=77002)\n    phase=np.linspace(0,8*np.pi,len(X));X[:,:4]+=.35*np.sin(phase[:,None]*np.arange(1,5)[None,:])\n    data[\'synthetic_process\']=(X,y,400,{\'source\':\'make_classification plus source-notebook process deformation\'})\n    digit=load_digits();mask=np.isin(digit.target,[3,8]);X=digit.data[mask];y=(digit.target[mask]==8).astype(int)\n    data[\'digits_3_vs_8\']=(X,y,80,{\'source\':\'real sklearn packaged handwritten digits; 3 versus 8\'})\n    return data\n\n\ndef nested_indices(y,seed,budgets):\n    """Each budget is a prefix of one balanced ordering, using labels only."""\n    rng=np.random.default_rng(seed);a=rng.permutation(np.flatnonzero(y==0));b=rng.permutation(np.flatnonzero(y==1))\n    order=[]\n    for k in range(max(map(len,[a,b]))):\n        if k<len(a):order.append(a[k])\n        if k<len(b):order.append(b[k])\n    return {n:np.array(order[:n]) for n in budgets}\n\n\ndef score_metrics(y,score,pred):\n    return {\'roc_auc\':roc_auc_score(y,score),\'average_precision\':average_precision_score(y,score),\n            \'balanced_accuracy\':balanced_accuracy_score(y,pred)}\n\n\ndef fit_kernel_method(X,y,Xtest,ytest,fit_idx,val_idx,name,mode,seed):\n    start=time.perf_counter();scale=StandardScaler().fit(X[fit_idx]);A=scale.transform(X[fit_idx]);B=scale.transform(X[val_idx])\n    yt,yv=y[fit_idx],y[val_idx];best=None\n    strengths=[1.] if mode==\'original\' else CONFIG[\'weight_strengths\'] if mode in [\'weighted\',\'sham\'] else [.5,1.,1.5]\n    Cs=CONFIG[\'original_C_values\'] if mode==\'original\' else CONFIG[\'weighted_C_values\']\n    for gamma in CONFIG[\'gamma_values\']:\n        K=kernel(A,A,name,gamma);cross=kernel(B,A,name,gamma)\n        for strength in strengths:\n            tr,va,_=spectral_map(K,cross,yt,mode,strength,seed)\n            for C in Cs:\n                clf=SVC(C=C,kernel=\'precomputed\',class_weight=\'balanced\')\n                clf.fit(tr,yt);scores=clf.decision_function(va);auc=roc_auc_score(yv,scores)\n                # Deterministic ties favor lower complexity: strength 0 first,\n                # smaller C first; no test outcomes enter this choice.\n                if best is None or auc>best[0]+1e-12:best=(auc,gamma,C,strength)\n    tuning_seconds=time.perf_counter()-start\n    scaler=StandardScaler().fit(X);A=scaler.transform(X);B=scaler.transform(Xtest)\n    K=kernel(A,A,name,best[1]);cross=kernel(B,A,name,best[1])\n    tr,te,diag=spectral_map(K,cross,y,mode,best[3],seed)\n    clf=SVC(C=best[2],kernel=\'precomputed\',class_weight=\'balanced\');clf.fit(tr,y)\n    metrics=score_metrics(ytest,clf.decision_function(te),clf.predict(te))\n    return {**metrics,**diag,\'chosen_gamma\':best[1],\'chosen_C\':best[2],\'chosen_strength\':best[3],\n            \'validation_AUC\':best[0],\'tuning_seconds\':tuning_seconds,\'total_seconds\':time.perf_counter()-start,\n            \'candidates\':len(CONFIG[\'gamma_values\'])*len(strengths)*len(Cs)}\n\n\ndef fit_standard(X,y,Xtest,ytest,fit_idx,val_idx,name):\n    start=time.perf_counter();scaler=StandardScaler().fit(X[fit_idx]);A=scaler.transform(X[fit_idx]);B=scaler.transform(X[val_idx])\n    best=None\n    if name==\'LogisticRegression\':\n        params=[{\'C\':C} for C in np.logspace(-3,3,27)]\n        make=lambda p:LogisticRegression(**p,class_weight=\'balanced\',max_iter=1000)\n    else:\n        params=[{\'max_leaf_nodes\':leaves,\'learning_rate\':rate,\'l2_regularization\':l2}\n                for leaves in [3,7,15] for rate in [.05,.1,.2] for l2 in [.1,1.,10.]]\n        make=lambda p:HistGradientBoostingClassifier(**p,min_samples_leaf=5,max_iter=60,\n                                                     early_stopping=False,random_state=42)\n    for p in params:\n        clf=make(p);clf.fit(A,y[fit_idx])\n        score=clf.decision_function(B) if hasattr(clf,\'decision_function\') else clf.predict_proba(B)[:,1]\n        auc=roc_auc_score(y[val_idx],score)\n        if best is None or auc>best[0]+1e-12:best=(auc,p)\n    tuning=time.perf_counter()-start;scaler=StandardScaler().fit(X);A=scaler.transform(X);B=scaler.transform(Xtest)\n    clf=make(best[1]);clf.fit(A,y);score=clf.decision_function(B) if hasattr(clf,\'decision_function\') else clf.predict_proba(B)[:,1]\n    return {**score_metrics(ytest,score,clf.predict(B)),\'validation_AUC\':best[0],\n            \'tuning_seconds\':tuning,\'total_seconds\':time.perf_counter()-start,\'candidates\':27,\n            \'standard_parameters\':json.dumps(best[1])}\n\n\ndef meaningful_checks():\n    rng=np.random.default_rng(982);X=rng.normal(size=(30,8));Z=rng.normal(size=(9,8));y=np.tile([0,1],15)\n    checks=[]\n    for name in KERNELS:\n        K=kernel(X,X,name,.05)\n        assert np.linalg.eigvalsh((K+K.T)/2).min()>-1e-9,name\n        for mode,strength in [(\'original\',1),(\'weighted\',0),(\'weighted\',1),(\'sham\',1),(\'power\',1.5)]:\n            tr,cross,diag=spectral_map(K,K,y,mode,strength,9)\n            err=np.linalg.norm(tr-cross)/np.linalg.norm(tr)\n            assert err<1e-8,(name,mode,err)\n            assert np.linalg.eigvalsh(tr).min()>-1e-8,(name,mode)\n        orig,_,_=spectral_map(K,kernel(Z,X,name,.05),y,\'original\',1,9)\n        zero,_,_=spectral_map(K,kernel(Z,X,name,.05),y,\'weighted\',0,9)\n        assert np.max(np.abs(orig-zero))<1e-12,name\n        # Separate finite raw-kernel reference matches source formulas at gamma=.05.\n        if name==\'ProcessUniform\':\n            d=X.shape[1];P=process_transform(X)\n            ref=d/(d+5)*np.exp(-.05*np.sum((P[:,None]-P[None,:])**2,axis=2))+5/(d+5)*(X@X.T/d)\n            assert np.max(np.abs(K-ref))<1e-12\n        checks.append({\'kernel\':name,\'PSD\':True,\'consistent_extension\':True,\'zero_weight_identity\':True})\n    return checks\n\n\ndef summarize(frame):\n    group=[\'dataset\',\'budget\',\'method\']\n    summary=frame.groupby(group).agg(auc_mean=(\'roc_auc\',\'mean\'),auc_sd=(\'roc_auc\',\'std\'),\n                                    balanced_accuracy_mean=(\'balanced_accuracy\',\'mean\'),\n                                    average_precision_mean=(\'average_precision\',\'mean\'),\n                                    seconds_mean=(\'total_seconds\',\'mean\'),runs=(\'seed\',\'count\')).reset_index()\n    factor=student_t.ppf(.975,summary.runs-1)*summary.auc_sd/np.sqrt(summary.runs)\n    summary[\'auc_CI_low\']=summary.auc_mean-factor;summary[\'auc_CI_high\']=summary.auc_mean+factor\n    paired=[];interactions=[]\n    for dataset in frame.dataset.unique():\n        for name in KERNELS:\n            sub=frame[(frame.dataset==dataset)&frame.method.isin([name,name+\' + weighting\'])]\n            pivot=sub.pivot(index=[\'seed\',\'budget\'],columns=\'method\',values=\'roc_auc\')\n            difference=pivot[name+\' + weighting\']-pivot[name]\n            for budget in CONFIG[\'budgets\']:\n                d=difference.xs(budget,level=\'budget\');half=student_t.ppf(.975,len(d)-1)*d.std(ddof=1)/np.sqrt(len(d))\n                paired.append({\'dataset\':dataset,\'kernel\':name,\'budget\':budget,\'mean_AUC_gain\':d.mean(),\n                               \'CI_low\':d.mean()-half,\'CI_high\':d.mean()+half,\'seed_win_fraction\':np.mean(d>0),\n                               \'pointwise_only\':True})\n            d=difference.unstack(\'budget\');scarce=d[CONFIG[\'scarce_budgets\']].mean(axis=1);large=d[256];interaction=scarce-large\n            row={\'dataset\':dataset,\'kernel\':name}\n            for key,values in [(\'scarce_gain\',scarce),(\'large_gain\',large),(\'scarcity_interaction\',interaction)]:\n                half=student_t.ppf(.975,len(values)-1)*values.std(ddof=1)/np.sqrt(len(values))\n                row[key]=values.mean();row[key+\'_CI_low\']=values.mean()-half;row[key+\'_CI_high\']=values.mean()+half\n            row[\'scarcity_specific_evidence\']=bool(row[\'scarce_gain_CI_low\']>.01 and\n                    row[\'scarcity_interaction_CI_low\']>0 and row[\'large_gain_CI_low\']>-.01 and row[\'large_gain_CI_high\']<.01)\n            interactions.append(row)\n    return summary,pd.DataFrame(paired),pd.DataFrame(interactions)\n\n\ndef main(out=\'spectral_scarcity_results\',seeds=12):\n    out=Path(out);out.mkdir(parents=True,exist_ok=True);CONFIG[\'seeds\']=seeds\n    # Written before any test evaluation. The same file enters the final bundle.\n    (out/\'protocol.json\').write_text(json.dumps(CONFIG,indent=2))\n    checks=meaningful_checks();all_data=datasets();rows=[];split_records=[];data_records=[]\n    methods=[(name,\'original\',name) for name in KERNELS]+[(name,\'weighted\',name+\' + weighting\') for name in KERNELS]\n    methods += [(\'Laplacian\',\'original\',\'Laplacian\'),(\'ProcessUniform\',\'power\',\'ProcessUniform + spectral power\'),\n                (\'ProcessUniform\',\'sham\',\'ProcessUniform + shuffled-label weighting\')]\n    started=time.perf_counter()\n    with threadpool_limits(limits=1):\n        for dataset,(X,y,test_n,meta) in all_data.items():\n            indices=np.arange(len(y));pool,test=train_test_split(indices,test_size=test_n,stratify=y,random_state=77003)\n            data_records.append({\'dataset\':dataset,\'source\':meta[\'source\'],\'rows\':len(y),\'features\':X.shape[1],\n                                 \'test_n\':test_n,\'sha256\':hashlib.sha256(X.tobytes()+y.tobytes()).hexdigest()})\n            np.savez_compressed(out/(dataset+\'_data.npz\'),X=X,y=y,pool=pool,test=test,\n                                hidden=meta.get(\'hidden_parameters\',np.empty((0,0))))\n            for seed in range(seeds):\n                ordering=nested_indices(y[pool],44000+seed,CONFIG[\'budgets\'])\n                for budget,loc in ordering.items():\n                    chosen=pool[loc];Xt,yt=X[chosen],y[chosen]\n                    fit,val=train_test_split(np.arange(budget),test_size=.25,stratify=yt,random_state=45000+seed)\n                    assert not np.intersect1d(chosen,test).size\n                    split_records.append({\'dataset\':dataset,\'seed\':seed,\'budget\':budget,\'available_indices\':chosen.tolist(),\n                                          \'inner_fit_local\':fit.tolist(),\'inner_validation_local\':val.tolist(),\'test_indices\':test.tolist()})\n                    for name,mode,label in methods:\n                        result=fit_kernel_method(Xt,yt,X[test],y[test],fit,val,name,mode,seed)\n                        rows.append({\'dataset\':dataset,\'seed\':seed,\'budget\':budget,\'method\':label,**result})\n                    for name in [\'LogisticRegression\',\'HistGradientBoosting\']:\n                        result=fit_standard(Xt,yt,X[test],y[test],fit,val,name)\n                        rows.append({\'dataset\':dataset,\'seed\':seed,\'budget\':budget,\'method\':name,**result})\n                # Checkpoint after complete nested learning curves per seed.\n                pd.DataFrame(rows).to_csv(out/\'all_runs.csv\',index=False)\n                print(f\'{dataset}: seed {seed+1}/{seeds}; {len(rows)} completed fits; {time.perf_counter()-started:.1f} s\',flush=True)\n    frame=pd.DataFrame(rows);assert (frame.candidates==27).all()\n    summary,paired,interaction=summarize(frame)\n    summary.to_csv(out/\'learning_curve_summary.csv\',index=False)\n    paired.to_csv(out/\'paired_weighting_gains.csv\',index=False)\n    interaction.to_csv(out/\'scarcity_interactions.csv\',index=False)\n    (out/\'splits.json\').write_text(json.dumps(split_records))\n    manifest={\'config\':CONFIG,\'python\':platform.python_version(),\'numpy\':np.__version__,\'scipy\':scipy.__version__,\n              \'sklearn\':sklearn.__version__,\'data\':data_records,\'checks\':checks,\'runtime_seconds\':time.perf_counter()-started,\n              \'kernel_source\':\'Kaggle_GPU_Spectral_Process_Kernel_Validation(2).ipynb cell 4, NumPy float64 translation\',\n              \'common_processing\':\'fit-only scaling for selection; full budget scaling for refit; centered mean-trace-normalized kernels\',\n              \'weighting\':\'three spectral bands, train-only target alignment, nonnegative shrunk band weights; valid out-of-sample map\',\n              \'extra_information\':\'none; hidden simulator metadata never passed to learners\',\n              \'multiplicity\':\'pointwise intervals across many exploratory comparisons; one prespecified primary dataset/kernel interaction\',\n              \'claims_excluded\':[\'quantum advantage\',\'RH proof\',\'particle production\',\'privileged test access\',\'clinical/device validation\']}\n    (out/\'manifest.json\').write_text(json.dumps(manifest,indent=2))\n    plt.rcParams.update({\'font.size\':10,\'axes.spines.top\':False,\'axes.spines.right\':False})\n    fig,axs=plt.subplots(1,3,figsize=(15,4.8),layout=\'constrained\')\n    colors={\'RBF\':\'#3f6b8a\',\'RBF + weighting\':\'#86b2c3\',\'ProcessUniform\':\'#b26228\',\n            \'ProcessUniform + weighting\':\'#e3a36a\',\'ProcessLamb-PSD + weighting\':\'#588451\',\n            \'HistGradientBoosting\':\'#865885\',\'LogisticRegression\':\'#333333\'}\n    for ax,dataset in zip(axs,frame.dataset.unique()):\n        for method,color in colors.items():\n            sub=summary[(summary.dataset==dataset)&(summary.method==method)].sort_values(\'budget\')\n            ax.plot(sub.budget,sub.auc_mean,\'o-\',label=method,color=color,linewidth=1.5)\n        ax.set(xlabel=\'Total labelled observations (including validation)\',ylabel=\'Test ROC-AUC\',title=dataset.replace(\'_\',\' \'))\n        ax.set_xscale(\'log\',base=2);ax.set_xticks(CONFIG[\'budgets\'],[str(x) for x in CONFIG[\'budgets\']]);ax.set_ylim(.45,1.01)\n    axs[0].legend(fontsize=7,loc=\'lower right\');fig.suptitle(f\'Observation removal | fixed test sets | {seeds} resampling seeds\',fontsize=15)\n    fig.savefig(out/\'learning_curves.png\',dpi=180);plt.close(fig)\n    fig,axs=plt.subplots(1,3,figsize=(15,4.5),layout=\'constrained\')\n    for ax,dataset in zip(axs,frame.dataset.unique()):\n        for j,name in enumerate(KERNELS):\n            sub=paired[(paired.dataset==dataset)&(paired.kernel==name)].sort_values(\'budget\')\n            x=np.arange(4)+(j-1.5)*.10\n            ax.errorbar(x,100*sub.mean_AUC_gain,yerr=np.array([100*(sub.mean_AUC_gain-sub.CI_low),100*(sub.CI_high-sub.mean_AUC_gain)]),\n                        marker=\'o\',linestyle=\'-\',label=name,linewidth=1)\n        ax.axhline(0,color=\'black\',linewidth=.8);ax.set_xticks(np.arange(4),[\'32\',\'64\',\'128\',\'256\'])\n        ax.set(xlabel=\'Labelled observation budget\',ylabel=\'Weighted minus original AUC (percentage points)\',title=dataset.replace(\'_\',\' \'))\n    axs[0].legend(fontsize=7);fig.suptitle(\'Paired gains | pointwise 95% t intervals conditional on fixed test sets\',fontsize=14)\n    fig.savefig(out/\'paired_gains.png\',dpi=180);plt.close(fig)\n    print(\'\\nSCARCITY INTERACTIONS\\n\'+interaction.to_string(index=False),flush=True)\n    return frame,summary,paired,interaction,manifest\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__);parser.add_argument(\'--out\',default=\'spectral_scarcity_results\')\n    parser.add_argument(\'--seeds\',type=int,default=12);args=parser.parse_args();main(args.out,args.seeds)\n', 'separation_navigation_experiment.py': '"""Fresh, reproducible separation tests: sensor scarcity, marginal likelihood,\nfrozen process kernels, zeta/resolvent controls, root counting and Ricci flow.\nRun with OPENBLAS_NUM_THREADS=1 OMP_NUM_THREADS=1 python this_file.py.\nRequires the two sibling engine files included in the downloadable bundle.\n"""\nimport os\nos.environ.setdefault(\'OPENBLAS_NUM_THREADS\',\'1\')\nos.environ.setdefault(\'OMP_NUM_THREADS\',\'1\')\nfrom pathlib import Path\nimport argparse, json, time, hashlib, platform\nimport numpy as np\nimport pandas as pd\nfrom scipy.special import bdtrc, digamma, logsumexp\nfrom scipy.integrate import quad, solve_ivp\nfrom scipy.stats import qmc, t as student_t\nfrom scipy.optimize import brentq\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import roc_auc_score, balanced_accuracy_score, average_precision_score\nfrom threadpoolctl import threadpool_limits\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nimport spectral_scarcity_experiment as previous\nimport fusion_zeta_experiment as fusion\n\nTIMES=np.array([.1,.25,.5,.75,1.,1.5,2.5,4.])\nSENSORS={1:[7],2:[0,7],4:[0,2,5,7],8:list(range(8))}\nCONFIG={\'version\':\'SeparationNavigation-v1\',\'seeds\':8,\'label_budgets\':[64,256],\n        \'sensor_time_budgets\':[1,2,4,8],\'sensor_times_seconds\':TIMES.tolist(),\n        \'fixed_sensor_indices\':SENSORS,\'development_cases\':1600,\'fresh_test_cases\':400,\n        \'sobol_bank_per_class\':16384,\'convergence_bank_per_class\':8192,\n        \'validation_fraction\':.25,\'primary_comparison\':\'ProcessUniform + weighting minus RBF\',\n        \'primary_budget\':256,\'primary_sensor_times\':8,\'meaningful_AUC_gain\':.01,\n        \'mass_controls\':[.05,.2,1.],\'linear_C_grid\':np.logspace(-3,3,9).tolist(),\n        \'root_rectangle\':[-.05,1.05,10.,65.],\n        \'warnings\':[\'Synthetic two-reservoir thermal model; not plasma MHD.\',\n                    \'Physics detector knows equations and parameter/noise distributions.\',\n                    \'Monte Carlo likelihood and contour count are approximate, not certified.\',\n                    \'Zeta-to-time mapping is a chosen representation, not a physical derivation.\',\n                    \'Ricci flow is a separate geometry demonstration, not fusion heating.\']}\n\ndef parameters(uniform,y):\n    """Five independent uniform nuisance variables, with declared class priors."""\n    n=(.7+.6*uniform[:,0])*1e20\n    tauE=np.where(y==0,1.2+1.2*uniform[:,1],2.4+1.2*uniform[:,1])\n    escape=np.where(y==0,1+3*uniform[:,2],.03+.27*uniform[:,2])\n    return np.column_stack([n,tauE,escape,.15+.35*uniform[:,3],12+6*uniform[:,4]])\n\ndef simulate(p,step=.01):\n    n,tauE,escape,slow,T0=p.T\n    C=3*n*1.602176634e-16;U=C*T0;A=np.zeros(len(p))\n    frames=[];boundary=False\n    capture=set(np.rint(TIMES/step).astype(int))\n    def rhs(u,a):\n        temp=u/C\n        pa=.25*n*n*previous.reactivity(temp)*3500*1.602176634e-16\n        return a/slow-u/tauE-5.35e-37*n*n*np.sqrt(temp),pa-a/slow-a/escape\n    for j in range(1,int(round(4/step))+1):\n        a,b=rhs(U,A);c,d=rhs(U+step*a/2,A+step*b/2)\n        e,f=rhs(U+step*c/2,A+step*d/2);g,h=rhs(U+step*e,A+step*f)\n        U+=step*(a+2*c+2*e+g)/6;A+=step*(b+2*d+2*f+h)/6\n        boundary=boundary or bool(np.any((U/C<=.2)|(U/C>=100)))\n        if j in capture:frames.append((U/C).copy())\n    if boundary:raise RuntimeError(\'Trajectory left the Bosch-Hale fit interval\')\n    temp=np.column_stack(frames)\n    logN=np.log(.25*n[:,None]**2*previous.reactivity(temp)/1e18)\n    mean=np.column_stack([temp,logN])\n    sd=np.column_stack([.25+.03*temp,np.full_like(temp,.10)])\n    return mean,sd\n\ndef likelihood(X,mean,sd,indices,chunk=64):\n    """Integrate nuisance variables by scrambled-Sobol prior quadrature.\n    Test-case metadata is never supplied. Each bank trajectory is independent.\n    """\n    mu=mean[:,indices];sigma=sd[:,indices];inv=1/(sigma*sigma)\n    linear=mu*inv\n    offset=np.sum(mu*mu*inv+2*np.log(sigma)+np.log(2*np.pi),axis=1)\n    result=[]\n    for begin in range(0,len(X),chunk):\n        x=X[begin:begin+chunk,indices]\n        logp=-.5*((x*x)@inv.T-2*x@linear.T+offset[None,:])\n        result.append(logsumexp(logp,axis=1)-np.log(len(mu)))\n    return np.concatenate(result)\n\ndef metric(y,s):\n    return {\'roc_auc\':float(roc_auc_score(y,s)),\n            \'balanced_accuracy\':float(balanced_accuracy_score(y,s>=0)),\n            \'average_precision\':float(average_precision_score(y,s))}\n\ndef zeta_array(s,terms=256,derivative=False):\n    s=np.asarray(s,dtype=complex);flat=s.ravel();k=np.arange(terms)\n    logn=np.log(k+1);w=(-1.)**k*bdtrc(k,terms,.5)\n    result=[];der=[]\n    for start in range(0,len(flat),512):\n        z=flat[start:start+512];ex=np.exp(-z[:,None]*logn)\n        eta=ex@w;power=np.exp((1-z)*np.log(2));den=1-power\n        result.append(eta/den)\n        if derivative:der.append((ex@(-logn*w))/den-eta*np.log(2)*power/den**2)\n    v=np.concatenate(result).reshape(s.shape)\n    return (v,np.concatenate(der).reshape(s.shape)) if derivative else v\n\ndef root_and_contour_checks(out):\n    grid=np.arange(10,65.0001,.025);v=np.array([fusion.hardy_z(t) for t in grid])\n    roots=np.array([brentq(fusion.hardy_z,a,b,xtol=1e-12)\n                    for a,b,fa,fb in zip(grid[:-1],grid[1:],v[:-1],v[1:]) if fa*fb<0])\n    residual=np.abs(zeta_array(.5+1j*roots,384))\n    pd.DataFrame({\'order\':np.arange(1,len(roots)+1),\'height\':roots,\'residual\':residual}).to_csv(out/\'ordered_zeros.csv\',index=False)\n    # Argument principle for xi; rectangle stays away from removable eta poles.\n    corners=[-.05+10j,1.05+10j,1.05+65j,-.05+65j,-.05+10j]\n    counts=[]\n    for terms in [192,256,384]:\n        value=0j;error=0\n        def ld(s):\n            z,d=zeta_array(np.array([s]),terms,True)\n            return 1/s+1/(s-1)-.5*np.log(np.pi)+.5*digamma(s/2)+d[0]/z[0]\n        for a,b in zip(corners[:-1],corners[1:]):\n            delta=b-a\n            fn=lambda t:ld(a+t*delta)*delta\n            real,er=quad(lambda t:fn(t).real,0,1,epsabs=1e-8,epsrel=1e-9,limit=600)\n            imag,ei=quad(lambda t:fn(t).imag,0,1,epsabs=1e-8,epsrel=1e-9,limit=600)\n            value+=real+1j*imag;error+=er+ei\n        count=value/(2j*np.pi)\n        counts.append({\'terms\':terms,\'count_real\':count.real,\'count_imag\':count.imag,\n                       \'quadrature_error_estimate\':error/(2*np.pi)})\n    pd.DataFrame(counts).to_csv(out/\'contour_counts.csv\',index=False)\n    assert len(roots)==14 and max(abs(r[\'count_real\']-len(roots)) for r in counts)<1e-6\n    return roots,{\'critical_line_roots\':len(roots),\'maximum_root_residual\':float(residual.max()),\n                  \'contour_count\':counts[-1][\'count_real\'],\'rigorous_certificate\':False}\n\ndef geometry_and_instability(out):\n    """Actual 2D conformal Ricci flow on a periodic torus, independent of plasma.\n    Metric g=a(dx^2+dy^2); a_t=Delta log(a). Discrete area is conserved.\n    """\n    n=32;dx=2*np.pi/n;u=np.arange(n)*dx\n    a=np.exp(2*(.15*np.cos(u[:,None])+.12*np.cos(u[None,:])))\n    def lap(z):return (np.roll(z,1,0)+np.roll(z,-1,0)+np.roll(z,1,1)+np.roll(z,-1,1)-4*z)/dx**2\n    rows=[];initial=a.copy()\n    for j in range(2001):\n        if j%100==0:\n            R=-lap(np.log(a))/a\n            rows.append({\'flow_time\':j*.001,\'area\':float(a.sum()*dx**2),\n                         \'scalar_curvature_rms\':float(np.sqrt(np.sum(R*R*a)/a.sum())),\n                         \'integrated_scalar_curvature\':float(np.sum(R*a)*dx**2),\n                         \'minimum_metric_density\':float(a.min()),\'maximum_metric_density\':float(a.max())})\n        if j<2000:a+=.001*lap(np.log(a))\n        assert np.all(a>0)\n    df=pd.DataFrame(rows);df.to_csv(out/\'ricci_flow.csv\',index=False)\n    fig,axes=plt.subplots(1,3,figsize=(12,3.6),layout=\'constrained\')\n    for ax,arr,title in zip(axes[:2],[initial,a],[\'Initial metric density\',\'After geometric Ricci flow\']):\n        im=ax.imshow(arr,vmin=initial.min(),vmax=initial.max(),origin=\'lower\',cmap=\'viridis\');ax.set_title(title);ax.set_xticks([]);ax.set_yticks([])\n    fig.colorbar(im,ax=axes[:2],shrink=.8)\n    axes[2].plot(df.flow_time,df.scalar_curvature_rms);axes[2].set(xlabel=\'Geometric flow time\',ylabel=\'Curvature RMS\',title=\'Smoothing; no heat-source model\')\n    fig.savefig(out/\'ricci_geometry.png\',dpi=160);plt.close(fig)\n    # Perturb each equilibrium along its largest-real-eigenvalue eigenvector.\n    eq=fusion.equilibria();eq.to_csv(out/\'thermal_equilibria.csv\',index=False)\n    mode_rows=[];trace_rows=[];C=3*1e20*1.602176634e-16;slow=.3;escape=3.;tauE=3.\n    def rhs(t,z):\n        T,A=z;pa=.25*1e40*previous.reactivity(T)*3500*1.602176634e-16\n        return [(A/slow-C*T/tauE-5.35e-37*1e40*np.sqrt(T))/C,pa-A/slow-A/escape]\n    for _,r in eq.iterrows():\n        T=r.T_keV;pa=.25*1e40*previous.reactivity(T)*3500*1.602176634e-16\n        A=pa/(1/slow+1/escape);reference=np.array([T,A]);scale=np.array([T,A])\n        h=1e-4;J=np.column_stack([(np.asarray(rhs(0,reference+scale*np.eye(2)[k]*h))-np.asarray(rhs(0,reference-scale*np.eye(2)[k]*h)))/(2*h)/scale for k in range(2)])\n        eig,V=np.linalg.eig(J);i=np.argmax(eig.real);direction=V[:,i].real;direction/=np.linalg.norm(direction)\n        grid=np.linspace(0,2,201)\n        sol=solve_ivp(rhs,(0,2),reference+1e-5*scale*direction,t_eval=grid,rtol=1e-11,atol=1e-8,max_step=.01)\n        norm=np.linalg.norm((sol.y.T-reference)/scale,axis=1)\n        fitted=np.polyfit(grid,np.log(norm),1)[0]\n        mode_rows.append({\'T_keV\':T,\'stability\':r.thermal_stability,\n                          \'growth_eigenvalue\':float(eig[i].real),\'measured_perturbation_log_slope\':float(fitted)})\n        trace_rows.extend({\'T_equilibrium\':T,\'time_s\':t,\'relative_perturbation\':d} for t,d in zip(grid,norm))\n    pd.DataFrame(mode_rows).to_csv(out/\'instability_separation.csv\',index=False)\n    pd.DataFrame(trace_rows).to_csv(out/\'instability_traces.csv\',index=False)\n    ignition=fusion.ignition_scan();ignition.to_csv(out/\'ignition_optimization.csv\',index=False)\n    return {\'ricci_area_relative_drift\':float(abs(df.area.iloc[-1]/df.area.iloc[0]-1)),\n            \'ricci_curvature_rms_ratio\':float(df.scalar_curvature_rms.iloc[-1]/df.scalar_curvature_rms.iloc[0]),\n            \'maximum_growth_rate_discrepancy\':float(max(abs(r[\'growth_eigenvalue\']-r[\'measured_perturbation_log_slope\']) for r in mode_rows))}\n\ndef representation_maps(indices,roots):\n    """Quadrature for actual time-ordered input channels; no synthetic filling\n    is treated as a measured sensor. Interpolation is the declared encoding q_x.\n    The zeta/resolvent mapping is explicit and arbitrary, not learned physics.\n    """\n    grid=np.linspace(0,1,257);du=1/256;weights=np.ones(257)*du;weights[[0,-1]]*=.5\n    times=TIMES[indices]/4\n    B=np.column_stack([np.interp(grid,times,np.eye(len(times))[:,i]) for i in range(len(times))])\n    window=np.sin(np.pi*grid);length=55.;j=np.arange(1,33)\n    basis=np.sqrt(2/length)*np.sin(np.pi*grid[:,None]*j)\n    z=zeta_array(.5+1j*(10+55*grid));z/=np.sqrt(np.sum(abs(z)**2*weights))\n    scrambled=z[np.random.default_rng(813).permutation(len(z))]\n    result={}\n    for label,carrier in [(\'Resolvent plain\',np.ones(len(grid),complex)),(\'Resolvent zeta\',z),(\'Resolvent scrambled\',scrambled)]:\n        raw=(B*(window*carrier*weights)[None,:].T).T@basis*length\n        # B is grid-by-sensor; raw is sensor-by-mode. Real part of complex Gram.\n        for mass in CONFIG[\'mass_controls\']:\n            mapped=2*np.pi*raw/((np.pi*j/length)**2+mass**2)[None,:]\n            feature=np.column_stack([mapped.real,mapped.imag]);d=len(times)\n            block=np.zeros((2*d,2*feature.shape[1]));block[:d,:feature.shape[1]]=feature;block[d:,feature.shape[1]:]=feature\n            result[(label,mass)]=block\n    fz=roots[:12]/roots[0];fu=np.linspace(fz.min(),fz.max(),len(fz))\n    gaps=np.diff(fz);fs=np.r_[fz[0],fz[0]+np.cumsum(np.random.default_rng(812).permutation(gaps))]\n    for label,freq in [(\'Zero frequencies\',fz),(\'Uniform frequencies\',fu),(\'Scrambled frequencies\',fs)]:\n        oscillation=np.column_stack([np.ones(len(grid)),np.cos(2*np.pi*grid[:,None]*freq),np.sin(2*np.pi*grid[:,None]*freq)])\n        feature=B.T@(oscillation*weights[:,None]);d=len(times)\n        block=np.zeros((2*d,2*feature.shape[1]));block[:d,:feature.shape[1]]=feature;block[d:,feature.shape[1]:]=feature\n        result[(label,1.)]=block\n    return result\n\ndef fit_representation(X,y,Xtest,ytest,fit,val,maps,label):\n    best=None;start=time.perf_counter()\n    scaler=StandardScaler().fit(X[fit]);A=scaler.transform(X[fit]);B=scaler.transform(X[val])\n    options=[(p,M) for (name,p),M in maps.items() if name==label]\n    for p,M in options:\n        tr=A@M;va=B@M;norm=np.sqrt(np.mean(np.sum(tr*tr,axis=1)));tr/=norm;va/=norm\n        for C in CONFIG[\'linear_C_grid\']:\n            clf=LogisticRegression(C=C,class_weight=\'balanced\',max_iter=1000).fit(tr,y[fit])\n            auc=roc_auc_score(y[val],clf.decision_function(va))\n            if best is None or auc>best[0]+1e-12:best=(auc,p,C)\n    M=maps[(label,best[1])];scaler=StandardScaler().fit(X);tr=scaler.transform(X)@M;te=scaler.transform(Xtest)@M\n    norm=np.sqrt(np.mean(np.sum(tr*tr,axis=1)));tr/=norm;te/=norm\n    clf=LogisticRegression(C=best[2],class_weight=\'balanced\',max_iter=1000).fit(tr,y)\n    score=clf.decision_function(te)\n    return {**metric(ytest,score),\'validation_AUC\':best[0],\'chosen_mass_or_frequency_scale\':best[1],\n            \'chosen_C\':best[2],\'candidates\':len(options)*len(CONFIG[\'linear_C_grid\']),\n            \'total_seconds\':time.perf_counter()-start}\n\ndef ci(a):\n    a=np.asarray(a,dtype=float);mean=float(a.mean())\n    if len(a)<2:return mean,mean,mean\n    radius=float(student_t.ppf(.975,len(a)-1)*a.std(ddof=1)/np.sqrt(len(a)))\n    return mean,mean-radius,mean+radius\n\ndef summarize(frame,out):\n    rows=[]\n    for (budget,sensors,method),f in frame.groupby([\'budget\',\'sensor_times\',\'method\']):\n        m,l,h=ci(f.roc_auc);rows.append({\'budget\':budget,\'sensor_times\':sensors,\'method\':method,\'mean_AUC\':m,\'low95\':l,\'high95\':h})\n    summary=pd.DataFrame(rows);summary.to_csv(out/\'separation_summary.csv\',index=False)\n    comparisons=[(\'ProcessUniform + weighting\',\'RBF\'),(\'ProcessUniform + weighting\',\'ProcessUniform\'),\n                 (\'Resolvent zeta\',\'Resolvent plain\'),(\'Resolvent zeta\',\'Resolvent scrambled\'),\n                 (\'Zero frequencies\',\'Uniform frequencies\'),(\'Zero frequencies\',\'Scrambled frequencies\')]\n    pairs=[]\n    for (budget,sensors),f in frame.groupby([\'budget\',\'sensor_times\']):\n        p=f.pivot(index=\'seed\',columns=\'method\',values=\'roc_auc\')\n        for a,b in comparisons:\n            m,l,h=ci(p[a]-p[b]);pairs.append({\'budget\':budget,\'sensor_times\':sensors,\'comparison\':a+\' minus \'+b,\'mean_gain\':m,\'low95\':l,\'high95\':h,\'meaningful_gain\':bool(l>.01)})\n    paired=pd.DataFrame(pairs);paired.to_csv(out/\'paired_comparisons.csv\',index=False)\n    selected=[\'RBF\',\'ProcessUniform + weighting\',\'LogisticRegression\',\'HistGradientBoosting\',\'Resolvent zeta\',\'Physics likelihood\']\n    fig,axes=plt.subplots(1,2,figsize=(12,4.5),layout=\'constrained\')\n    for ax,budget in zip(axes,CONFIG[\'label_budgets\']):\n        for method in selected:\n            f=summary[(summary.budget==budget)&(summary.method==method)].sort_values(\'sensor_times\')\n            ax.plot(f.sensor_times,f.mean_AUC,marker=\'o\',label=method)\n        ax.set(xlabel=\'Observed time points (temperature + neutron rate)\',ylabel=\'Test AUROC\',title=f\'{budget} labelled cases\',xticks=[1,2,4,8],ylim=(.45,1.01));ax.grid(alpha=.2)\n    axes[-1].legend(fontsize=8,loc=\'lower right\');fig.savefig(out/\'separation_curves.png\',dpi=180);plt.close(fig)\n    return summary,paired\n\ndef refine_physics_likelihood(out_path=\'separation_navigation_results\'):\n    """Post-run numerical validation prompted by 8192/16384 decision changes.\n    Fixed resolutions; test labels are used only for final reporting, never for\n    choosing bank size, model parameters, or the decision rule.\n    """\n    out=Path(out_path);sizes=[16384,32768,65536,131072]\n    (out/\'likelihood_refinement_protocol.json\').write_text(json.dumps({\n        \'purpose\':\'Numerical convergence extension, after the main run\',\n        \'sizes_per_class\':sizes,\'sensor_times\':8,\'bank_seeds\':[2026100210,2026100211],\n        \'selection\':\'No resolution selected by test performance; all sizes reported\',\n        \'extra_information\':\'Exact simulator prior and observation noise model\'},indent=2))\n    data=np.load(out/\'fresh_fusion_data.npz\');X=data[\'X\'][data[\'test\']];y=data[\'y\'][data[\'test\']]\n    scores={n:[] for n in sizes}\n    for cls in [0,1]:\n        uniform=qmc.Sobol(5,scramble=True,seed=2026100210+cls).random_base2(17)\n        mu,sd=simulate(parameters(uniform,np.full(len(uniform),cls)))\n        for n in sizes:scores[n].append(likelihood(X,mu[:n],sd[:n],list(range(16))))\n        print(f\'High-resolution likelihood integration: class {cls} complete.\',flush=True)\n    rows=[];last=None;all_scores={}\n    for n in sizes:\n        s=scores[n][1]-scores[n][0];all_scores[str(n)]=s\n        row={\'bank_per_class\':n,**metric(y,s)}\n        if last is not None:\n            row.update({\'decision_agreement_with_previous\':float(np.mean((s>=0)==(last>=0))),\n                        \'median_absolute_log_ratio_change\':float(np.median(abs(s-last)))})\n        rows.append(row);last=s\n    pd.DataFrame(rows).to_csv(out/\'likelihood_high_resolution.csv\',index=False)\n    pd.DataFrame({\'label\':y,**all_scores}).to_csv(out/\'likelihood_high_resolution_scores.csv\',index=False)\n    manifest_path=out/\'manifest.json\'\n    if manifest_path.exists():\n        manifest=json.loads(manifest_path.read_text())\n        manifest[\'high_resolution_likelihood\']=pd.DataFrame(rows).fillna(\'\').to_dict(\'records\')\n        manifest[\'current_driver_sha256\']=hashlib.sha256(Path(__file__).read_bytes()).hexdigest()\n        manifest_path.write_text(json.dumps(manifest,indent=2))\n    fig,axes=plt.subplots(1,2,figsize=(11,4),layout=\'constrained\')\n    for temp,f in pd.read_csv(out/\'instability_traces.csv\').groupby(\'T_equilibrium\'):\n        axes[0].semilogy(f.time_s,f.relative_perturbation,label=f\'{temp:.3f} keV equilibrium\')\n    axes[0].set(xlabel=\'Time (s)\',ylabel=\'Relative perturbation size\',title=\'Thermal growth versus decay\')\n    axes[0].legend(fontsize=9);axes[0].grid(alpha=.2)\n    df=pd.DataFrame(rows);axes[1].plot(df.bank_per_class,df.roc_auc,marker=\'o\')\n    axes[1].set_xscale(\'log\',base=2);axes[1].set(xlabel=\'Prior trajectories per class\',ylabel=\'Fixed-test AUROC\',title=\'Likelihood integration convergence\')\n    axes[1].set_xticks(df.bank_per_class,labels=[\'16,384\',\'32,768\',\'65,536\',\'131,072\'],rotation=15);axes[1].grid(alpha=.2)\n    fig.savefig(out/\'stability_and_integration.png\',dpi=180);plt.close(fig)\n    return pd.DataFrame(rows)\n\ndef main(out_path=\'separation_navigation_results\',seeds=8):\n    out=Path(out_path);out.mkdir(parents=True,exist_ok=True);CONFIG[\'seeds\']=seeds\n    (out/\'protocol.json\').write_text(json.dumps(CONFIG,indent=2)) # BEFORE test evaluation\n    start=time.perf_counter();roots,root_checks=root_and_contour_checks(out)\n    geometry_checks=geometry_and_instability(out)\n    print(\'Root counting, perturbation separation and geometric flow finished.\',flush=True)\n    rng=np.random.default_rng(2026100201);y=np.tile([0,1],1000);rng.shuffle(y)\n    p=parameters(rng.random((2000,5)),y);mean,sd=simulate(p);X=mean+rng.normal(size=mean.shape)*sd\n    pool,test=train_test_split(np.arange(2000),test_size=400,stratify=y,random_state=20261002)\n    np.savez_compressed(out/\'fresh_fusion_data.npz\',X=X,y=y,development=pool,test=test,hidden_metadata=p,latent_mean=mean,noise_sd=sd)\n    check=np.arange(0,2000,17);finer,_=simulate(p[check],step=.005)\n    step_error=float(np.max(abs(mean[check]-finer)))\n    bank=[]\n    for cls in [0,1]:\n        uniform=qmc.Sobol(5,scramble=True,seed=2026100210+cls).random_base2(14)\n        metadata=parameters(uniform,np.full(len(uniform),cls));mu,sigma=simulate(metadata);bank.append((mu,sigma))\n        np.savez_compressed(out/f\'physics_prior_bank_class_{cls}.npz\',mean=mu,sd=sigma)\n    oracle={};oracle_small={};alltime={}\n    for count,idx in SENSORS.items():\n        indices=idx+[i+8 for i in idx]\n        scores=[likelihood(X,mu,sigma,indices) for mu,sigma in bank]\n        small=[likelihood(X,mu[:8192],sigma[:8192],indices) for mu,sigma in bank]\n        oracle[count]=scores[1]-scores[0];oracle_small[count]=small[1]-small[0]\n        print(f\'Physics marginal likelihood: {count} time points complete.\',flush=True)\n    for i in range(8):\n        s=[likelihood(X,mu,sigma,[i,i+8]) for mu,sigma in bank];alltime[i]=s[1]-s[0]\n    # Convergence bank with independent scrambled Sobol points, full sensors.\n    independent=[]\n    for cls in [0,1]:\n        uniform=qmc.Sobol(5,scramble=True,seed=2026100250+cls).random_base2(14)\n        mu,sigma=simulate(parameters(uniform,np.full(len(uniform),cls)))\n        independent.append(likelihood(X[test],mu,sigma,list(range(16))))\n    independent_score=independent[1]-independent[0]\n    convergence=[]\n    for count in SENSORS:\n        convergence.append({\'sensor_times\':count,\'AUC_8192_bank\':roc_auc_score(y[test],oracle_small[count][test]),\n                            \'AUC_16384_bank\':roc_auc_score(y[test],oracle[count][test]),\n                            \'decision_agreement\':np.mean((oracle_small[count][test]>=0)==(oracle[count][test]>=0)),\n                            \'median_absolute_log_ratio_change\':np.median(abs(oracle_small[count][test]-oracle[count][test]))})\n    pd.DataFrame(convergence).to_csv(out/\'likelihood_convergence.csv\',index=False)\n    pd.DataFrame({\'test_index\':test,\'label\':y[test],\'independent_bank_log_ratio\':independent_score,\n                  **{f\'log_ratio_{c}_timepoints\':oracle[c][test] for c in SENSORS}}).to_csv(out/\'physics_test_scores.csv\',index=False)\n    rows=[];splits=[]\n    kernel_methods=[(n,\'original\',n) for n in previous.KERNELS]+[(\'ProcessUniform\',\'weighted\',\'ProcessUniform + weighting\'),(\'RBF\',\'weighted\',\'RBF + weighting\'),(\'Laplacian\',\'original\',\'Laplacian\')]\n    repnames=[\'Resolvent plain\',\'Resolvent zeta\',\'Resolvent scrambled\',\'Zero frequencies\',\'Uniform frequencies\',\'Scrambled frequencies\']\n    maps={count:representation_maps(idx,roots) for count,idx in SENSORS.items()}\n    for seed in range(seeds):\n        r=np.random.default_rng(52000+seed);orders=[r.permutation(pool[y[pool]==cls]) for cls in [0,1]]\n        for budget in CONFIG[\'label_budgets\']:\n            selected=np.r_[orders[0][:budget//2],orders[1][:budget//2]];r.shuffle(selected)\n            fit,val=train_test_split(np.arange(budget),test_size=.25,stratify=y[selected],random_state=53000+seed)\n            splits.append({\'seed\':seed,\'budget\':budget,\'selected\':selected.tolist(),\'fit\':selected[fit].tolist(),\'validation\':selected[val].tolist()})\n            assert not set(selected)&set(test)\n            for count,idx in SENSORS.items():\n                indices=idx+[i+8 for i in idx];A=X[selected][:,indices];B=X[test][:,indices];ys=y[selected];yt=y[test]\n                base={\'seed\':seed,\'budget\':budget,\'sensor_times\':count}\n                for name,mode,label in kernel_methods:\n                    result=previous.fit_kernel_method(A,ys,B,yt,fit,val,name,mode,seed)\n                    rows.append({**base,\'method\':label,**result})\n                for name in [\'LogisticRegression\',\'HistGradientBoosting\']:\n                    rows.append({**base,\'method\':name,**previous.fit_standard(A,ys,B,yt,fit,val,name)})\n                for label in repnames:\n                    rows.append({**base,\'method\':label,**fit_representation(A,ys,B,yt,fit,val,maps[count],label)})\n                rows.append({**base,\'method\':\'Physics likelihood\',**metric(yt,oracle[count][test]),\'candidates\':0})\n                if count==1:\n                    # Additional navigation test: choose a time using validation labels only.\n                    best=max(range(8),key=lambda i:roc_auc_score(y[selected[val]],alltime[i][selected[val]]))\n                    rows.append({**base,\'method\':\'Physics validation-selected time\',**metric(yt,alltime[best][test]),\n                                 \'chosen_time_index\':best,\'chosen_time_s\':TIMES[best],\'candidates\':8})\n                pd.DataFrame(rows).to_csv(out/\'all_runs.csv\',index=False)\n            print(f\'Seed {seed+1}/{seeds}, label budget {budget}: all separation methods complete.\',flush=True)\n    frame=pd.DataFrame(rows);summary,paired=summarize(frame,out)\n    (out/\'splits.json\').write_text(json.dumps(splits))\n    tests=previous.meaningful_checks()\n    for maps_per_count in maps.values():\n        for matrix in maps_per_count.values():\n            gram=matrix@matrix.T;assert np.linalg.eigvalsh(gram).min()>-1e-7*max(np.linalg.norm(gram),1)\n    manifest={\'config\':CONFIG,\'elapsed_seconds\':time.perf_counter()-start,\'final_evaluations\':len(frame),\n              \'fitted_learners\':int((~frame.method.str.startswith(\'Physics\')).sum()),\n              \'root_checks\':root_checks,\'geometry_and_stability_checks\':geometry_checks,\n              \'maximum_RK4_half_step_mean_difference\':step_error,\n              \'physics_full_sensor_independent_bank_AUC\':float(roc_auc_score(y[test],independent_score)),\n              \'physics_full_sensor_bank_AUC\':float(roc_auc_score(y[test],oracle[8][test])),\n              \'source_engine_sha256\':hashlib.sha256(Path(previous.__file__).read_bytes()).hexdigest(),\n              \'structural_checks\':tests,\'Python\':platform.python_version(),\n              \'inference\':\'Paired t intervals over training resamples, conditional on one fixed test set; exploratory comparisons.\'}\n    (out/\'manifest.json\').write_text(json.dumps(manifest,indent=2))\n    print(json.dumps({\'finished\':True,\'elapsed_seconds\':manifest[\'elapsed_seconds\'],\'final_evaluations\':len(frame)}),flush=True)\n    return frame,summary,paired,manifest\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser();parser.add_argument(\'--out\',default=\'separation_navigation_results\');parser.add_argument(\'--seeds\',type=int,default=8)\n    parser.add_argument(\'--refine-only\',action=\'store_true\')\n    args=parser.parse_args()\n    with threadpool_limits(limits=1):\n        if not args.refine_only:main(args.out,args.seeds)\n        print(refine_physics_likelihood(args.out).to_string(index=False),flush=True)\n', 'kernel_family_benchmark.py': '"""New paired kernel-family pilot, using the exact October engines as siblings.\n\npython kernel_family_benchmark.py --mode quick --out results\nNo downloaded data needed. Prediction runs on CPU; optional GPU timing is separate.\n"""\nimport os\nos.environ.setdefault(\'OPENBLAS_NUM_THREADS\', \'1\')\nos.environ.setdefault(\'OMP_NUM_THREADS\', \'1\')\nimport argparse\nimport hashlib\nimport json\nimport platform\nimport time\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport scipy\nimport sklearn\nfrom scipy.stats import t as student_t\nfrom sklearn.datasets import load_digits, make_classification, make_moons\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.preprocessing import StandardScaler\nfrom threadpoolctl import threadpool_limits\nimport spectral_scarcity_experiment as old\nimport separation_navigation_experiment as sep\n\nVERSION = \'KernelFamily-20261002-v1\'\nNEW_DATA_SEED = 2026100291\nSPLIT_SEED = 2026100292\nROOTS = np.array([14.1347251417347,21.0220396387716,25.0108575801457,\n 30.4248761258595,32.9350615877392,37.5861781588257,40.9187190121475,\n 43.3270732809150,48.0051508811672,49.7738324776723,52.9703214777145,56.4462476970634])\n\n\ndef interval(a):\n    a=np.asarray(a,float);m=float(a.mean())\n    if len(a)<2:return m,None,None\n    r=float(student_t.ppf(.975,len(a)-1)*a.std(ddof=1)/np.sqrt(len(a)))\n    return m,m-r,m+r\n\n\ndef arithmetic_maps(indices, cutoff=128):\n    """New exploratory Lambda carrier, smooth zero-free envelope and shuffled\n    arithmetic coefficients. Exact October encoding/operator/feature layout.\n    The smooth carrier sum (log n)/(2 sqrt n) is a deliberately declared\n    prime-power-density approximation, not the Riemann-von Mangoldt law.\n    """\n    grid=np.linspace(0,1,257);u=10+55*grid;length=55.\n    weights=np.ones(257)/256;weights[[0,-1]]*=.5\n    times=sep.TIMES[indices]/4\n    B=np.column_stack([np.interp(grid,times,np.eye(len(times))[:,i]) for i in range(len(times))])\n    n=np.arange(2,cutoff+1);lam=np.zeros(cutoff+1)\n    prime=np.ones(cutoff+1,bool);prime[:2]=False\n    for p in range(2,cutoff+1):\n        if prime[p]:\n            prime[p*p::p]=False\n            power=p\n            while power<=cutoff:lam[power]=np.log(p);power*=p\n    cutoff_window=.5*(1+np.cos(np.pi*n/(cutoff+1)))\n    common=cutoff_window/np.sqrt(n)\n    phase=np.exp(-1j*u[:,None]*np.log(n)[None,:])\n    shuffled=np.random.default_rng(2026100293).permutation(lam[n])\n    carriers={\'Resolvent prime-Lambda\':phase@(lam[n]*common),\n              \'Resolvent randomized-Lambda\':phase@(shuffled*common),\n              \'Resolvent smooth-envelope\':phase@(.5*np.log(n)*common)}\n    j=np.arange(1,33);basis=np.sqrt(2/length)*np.sin(np.pi*grid[:,None]*j)\n    out={}\n    for label,carrier in carriers.items():\n        carrier=carrier/np.sqrt(np.sum(abs(carrier)**2*weights))\n        raw=(B*(np.sin(np.pi*grid)*carrier*weights)[:,None]).T@basis*length\n        for mass in sep.CONFIG[\'mass_controls\']:\n            mapped=2*np.pi*raw/((np.pi*j/length)**2+mass**2)[None,:]\n            feature=np.column_stack([mapped.real,mapped.imag]);d=len(indices)\n            M=np.zeros((2*d,2*feature.shape[1]));M[:d,:feature.shape[1]]=feature;M[d:,feature.shape[1]:]=feature\n            out[label,mass]=M\n    return out\n\n\ndef checks():\n    result={\'frozen_weighting\':old.meaningful_checks()}\n    rng=np.random.default_rng(2026100294);A=rng.normal(size=(40,16))\n    maps=sep.representation_maps(list(range(8)),ROOTS)\n    maps.update(arithmetic_maps(list(range(8))))\n    entries=[]\n    for (name,mass),M in maps.items():\n        if not name.startswith(\'Resolvent\'):continue\n        F=A@M;K=F@F.T;e=np.linalg.eigvalsh(K)\n        relative_min=float(e.min()/max(e.max(),1))\n        assert relative_min>-1e-10,(name,mass,relative_min)\n        # Re-encoding identical rows must reproduce their training cross-Gram.\n        G=(A.copy()@M)@F.T\n        assert np.allclose(K,G,rtol=1e-12,atol=1e-8)\n        entries.append({\'name\':name,\'mass\':mass,\'relative_min_eigenvalue\':relative_min,\'heldout_extension\':True})\n    result[\'resolvent\']=entries\n    # Independent spatial finite-difference operator, with positive mass squared.\n    length=55;n=160;h=length/(n+1);m=.2\n    operator=(2*np.eye(n)-np.eye(n,k=1)-np.eye(n,k=-1))/h**2+m*m*np.eye(n)\n    e=np.linalg.eigvalsh(operator)\n    assert e.min()>m*m\n    result[\'positive_operator_min_eigenvalue\']=float(e.min())\n    # Carrier independent mass eigenvalues and common unitary invariance.\n    F=rng.normal(size=(20,32))+1j*rng.normal(size=(20,32))\n    K=F@F.conj().T;phase=np.exp(.7j)\n    result[\'common_rotation_relative_error\']=float(np.linalg.norm((phase*F)@(phase*F).conj().T-K)/np.linalg.norm(K))\n    assert result[\'common_rotation_relative_error\']<1e-12\n    # Quadrature convergence under fixed encoding/mass/carrier; no test tuning.\n    def map_at(points):\n        g=np.linspace(0,1,points);w=np.ones(points)/(points-1);w[[0,-1]]*=.5\n        B=np.column_stack([np.interp(g,sep.TIMES/4,np.eye(8)[:,i]) for i in range(8)])\n        j=np.arange(1,33);z=sep.zeta_array(.5+1j*(10+55*g));z/=np.sqrt(np.sum(abs(z)**2*w))\n        raw=(B*(np.sin(np.pi*g)*z*w)[:,None]).T@(np.sqrt(2/55)*np.sin(np.pi*g[:,None]*j))*55\n        return 2*np.pi*raw/((np.pi*j/55)**2+.2**2)\n    ms=[map_at(n) for n in [257,513,1025]]\n    result[\'zeta_quadrature_relative_map_change\']={\n      \'257_to_513\':float(np.linalg.norm(ms[0]-ms[1])/np.linalg.norm(ms[1])),\n      \'513_to_1025\':float(np.linalg.norm(ms[1]-ms[2])/np.linalg.norm(ms[2]))}\n    return result\n\n\ndef optional_csv(path,target,drop_columns):\n    if not target:raise ValueError(\'CSV requires --target; no outcome guessing is permitted\')\n    df=pd.read_csv(path)\n    if target not in df:raise ValueError(f\'Missing target column: {target}\')\n    df=df.dropna(subset=[target]);values=list(pd.unique(df[target]))\n    if len(values)!=2:raise ValueError(\'This harness requires a binary target\')\n    y=(df[target]==sorted(values,key=str)[1]).to_numpy(dtype=int)\n    X=df.drop(columns=[target]+drop_columns)\n    # Never fit imputation or encode unknown chronology using the full file.\n    if not all(pd.api.types.is_numeric_dtype(X[c]) for c in X):\n        raise ValueError(\'CSV features must be numeric; supply a separate train-only categorical pipeline\')\n    if not np.isfinite(X.to_numpy(float)).all():raise ValueError(\'CSV needs finite numeric features\')\n    if min(np.bincount(y))<150:raise ValueError(\'Need at least 150 cases in each class for fixed holdout/budgets\')\n    return X.to_numpy(float),y\n\n\ndef write_tables(rows,out):\n    frame=pd.DataFrame(rows);frame.to_csv(out/\'all_runs.csv\',index=False)\n    keys=[\'study\',\'dataset\',\'budget\',\'sensor_times\',\'method\',\'protocol\']\n    summary=[]\n    for group,f in frame.groupby(keys,dropna=False):\n        record=dict(zip(keys,group));m,l,h=interval(f.roc_auc)\n        summary.append({**record,\'mean_auc\':m,\'ci_low\':l,\'ci_high\':h,\'seeds\':len(f),\n                        \'mean_seconds\':float(f.total_seconds.mean())})\n    pd.DataFrame(summary).to_csv(out/\'summary.csv\',index=False)\n    pairs=[]\n    for group,f in frame.groupby([\'study\',\'dataset\',\'budget\',\'sensor_times\',\'protocol\'],dropna=False):\n        p=f.pivot(index=\'seed\',columns=\'method\',values=\'roc_auc\')\n        comparisons=[(n+\' + weighting\',n) for n in old.KERNELS]\n        comparisons += [(\'ProcessUniform + weighting\',\'RBF\'),(\'ProcessUniform + sham\',\'ProcessUniform + weighting\'),\n          (\'Resolvent zeta\',\'Resolvent plain\'),(\'Resolvent zeta\',\'Resolvent scrambled\'),\n          (\'Resolvent prime-Lambda\',\'Resolvent randomized-Lambda\'),(\'Resolvent prime-Lambda\',\'Resolvent smooth-envelope\')]\n        for a,b in comparisons:\n            if a not in p or b not in p:continue\n            gains=(p[a]-p[b]).dropna();m,l,h=interval(gains)\n            pairs.append(dict(zip([\'study\',\'dataset\',\'budget\',\'sensor_times\',\'protocol\'],group))|\n              {\'comparison\':a+\' minus \'+b,\'mean_gain\':m,\'ci_low\':l,\'ci_high\':h,\'seeds\':len(gains),\n               \'meaningful_gain_lower_bound_gt_001\':l is not None and l>.01})\n    pd.DataFrame(pairs).to_csv(out/\'paired_comparisons.csv\',index=False)\n    long=[]\n    for r in rows:\n        for metric in [\'roc_auc\',\'average_precision\',\'balanced_accuracy\',\'total_seconds\']:\n            long.append({\'dataset\':r[\'dataset\'],\'task\':\'classification\' if metric!=\'total_seconds\' else \'timing\',\n              \'protocol\':r[\'protocol\'],\'model\':r[\'method\'],\'metric\':metric if metric!=\'total_seconds\' else \'runtime_sec\',\n              \'value\':r[metric],\'seed\':r[\'seed\'],\'n\':r[\'test_n\'],\'hardware\':r[\'hardware\'],\n              \'software\':r[\'software\'],\'run_id\':VERSION,\'source\':\'executed-new-pilot\',\n              \'notes\':\'Fixed test population; training-resampling seeds; exploratory pointwise intervals\'})\n    pd.DataFrame(long).to_csv(out/\'benchmark_input.csv\',index=False)\n\n\ndef main(out=\'results\',mode=\'quick\',seeds=None,csv_path=None,target=None,drop_columns=None):\n    out=Path(out);out.mkdir(parents=True,exist_ok=True)\n    seeds=seeds or (5 if mode==\'quick\' else 12)\n    budgets=[64,256] if mode==\'quick\' else [32,64,128,256]\n    sensor_budgets=[2,8] if mode==\'quick\' else [1,2,4,8]\n    hardware=platform.processor() or platform.machine()\n    environment={\'python\':platform.python_version(),\'numpy\':np.__version__,\'scipy\':scipy.__version__,\n      \'pandas\':pd.__version__,\'sklearn\':sklearn.__version__,\'platform\':platform.platform(),\n      \'hardware\':hardware,\'cpu_count\':os.cpu_count(),\'prediction_device\':\'CPU\',\n      \'threadpools\':1}\n    software=\';\'.join(f\'{k}={environment[k]}\' for k in [\'numpy\',\'scipy\',\'sklearn\'])\n    sources={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in Path(__file__).parent.glob(\'*.py\')}\n    protocol={\'version\':VERSION,\'mode\':mode,\'seeds\':seeds,\'label_budgets\':budgets,\n      \'sensor_budgets\':sensor_budgets,\'data_seed\':NEW_DATA_SEED,\'split_seed\':SPLIT_SEED,\n      \'validation_fraction\':.25,\'refit_including_validation_labels\':True,\'holdout_size\':.25,\n      \'meaningful_auc_gain\':.01,\'frozen_weighting_config\':old.CONFIG,\'resolvent_config\':sep.CONFIG,\n      \'prime_cutoff\':128,\'arithmetic_maps\':\'Exploratory extension, not October reproduction\',\n      \'mass_definition\':\'operator eigenvalue=(j*pi/55)^2+mass^2, exact engine convention\',\n      \'sensor_noise\':\'same equations/noise as October; independently generated cases\',\n      \'information\':\'Every learner gets labelled observations only; hidden parameters excluded\',\n      \'limitations\':[\'Intervals conditional on fixed test cases; no population significance\',\n        \'Pointwise exploratory intervals, unadjusted for multiple comparisons\',\n        \'Exact engines use different C/strength grids; comparisons are tuned procedures\',\n        \'Linear resolvent classifiers vs SVC kernels differ in estimator and representation\',\n        \'Synthetic fusion; packaged digits are reused data with a new exploratory partition\',\n        \'Optional CSV supports IID row splits only; not chronological/grouped deployment evidence\'],\n      \'csv_path\':str(csv_path) if csv_path else None,\'target\':target,\'drop_columns\':drop_columns or []}\n    # Written before accessing test metrics; no test-driven grid edits.\n    (out/\'protocol.json\').write_text(json.dumps(protocol,indent=2))\n    (out/\'environment.json\').write_text(json.dumps(environment,indent=2))\n    (out/\'source_hashes.json\').write_text(json.dumps(sources,indent=2))\n    start=time.perf_counter();rows=[];splits=[]\n    with threadpool_limits(limits=1):\n        (out/\'correctness_checks.json\').write_text(json.dumps(checks(),indent=2))\n        rng=np.random.default_rng(NEW_DATA_SEED);y=np.tile([0,1],1000);rng.shuffle(y)\n        metadata=sep.parameters(rng.random((2000,5)),y);mean,sd=sep.simulate(metadata)\n        fusion_X=mean+rng.normal(size=mean.shape)*sd\n        # Hidden simulator parameters deliberately never passed to fit routines.\n        np.savez_compressed(out/\'new_fusion_data.npz\',X=fusion_X,y=y)\n        Xp,yp=make_classification(n_samples=1600,n_features=24,n_informative=12,\n          n_redundant=4,weights=[.72,.28],class_sep=1.,flip_y=.03,random_state=NEW_DATA_SEED)\n        Xm,ym=make_moons(n_samples=1200,noise=.22,random_state=NEW_DATA_SEED)\n        digit=load_digits();mask=np.isin(digit.target,[3,8])\n        tasks={\'synthetic_process_fresh\':(Xp,yp),\'moons_fresh\':(Xm,ym),\n          \'digits_3_vs_8_repartition\':(digit.data[mask],(digit.target[mask]==8).astype(int))}\n        if csv_path:tasks[\'CSV_\'+Path(csv_path).stem]=optional_csv(csv_path,target,drop_columns or [])\n        methods=[(n,\'original\',n) for n in old.KERNELS]\n        methods += [(n,\'weighted\',n+\' + weighting\') for n in old.KERNELS]\n        methods += [(\'ProcessUniform\',\'sham\',\'ProcessUniform + sham\'),(\'Laplacian\',\'original\',\'Laplacian\')]\n        def run_task(dataset,X,y,study,sensors,maps=None,shared_split=None):\n            if shared_split is None:\n                pool,test=train_test_split(np.arange(len(y)),test_size=.25,stratify=y,random_state=SPLIT_SEED)\n            else:pool,test=shared_split\n            np.savez_compressed(out/(dataset+\'_data.npz\'),X=X,y=y,development=pool,test=test)\n            repnames=sorted({k[0] for k in maps}) if maps else []\n            protocol_id=f\'{VERSION};{study};labels={{budget}};sensors={sensors};data={NEW_DATA_SEED};split={SPLIT_SEED};grid27\'\n            for seed in range(seeds):\n                nested=old.nested_indices(y[pool],50000+seed,budgets)\n                for budget,local in nested.items():\n                    if len(local)<budget:raise ValueError(f\'{dataset}: insufficient development cases for budget {budget}\')\n                    chosen=pool[local];XX=X[chosen];yy=y[chosen]\n                    fit,val=train_test_split(np.arange(budget),test_size=.25,stratify=yy,random_state=61000+seed)\n                    if np.intersect1d(chosen,test).size:raise AssertionError(\'Test/development leakage\')\n                    splits.append({\'dataset\':dataset,\'sensors\':sensors,\'seed\':seed,\'budget\':budget,\n                      \'labelled_indices\':chosen.tolist(),\'validation_indices\':chosen[val].tolist(),\'test_indices\':test.tolist()})\n                    def append(label,r):\n                        record={\'study\':study,\'dataset\':dataset,\'seed\':seed,\'budget\':budget,\'sensor_times\':sensors,\n                          \'method\':label,\'protocol\':protocol_id.format(budget=budget),\'test_n\':len(test),\n                          \'hardware\':hardware,\'software\':software,**r}\n                        for k in [\'roc_auc\',\'average_precision\',\'balanced_accuracy\',\'total_seconds\']:\n                            if not np.isfinite(record[k]):raise AssertionError((label,k))\n                        rows.append(record)\n                    for name,weight,label in methods:\n                        append(label,old.fit_kernel_method(XX,yy,X[test],y[test],fit,val,name,weight,seed))\n                    for label in [\'LogisticRegression\',\'HistGradientBoosting\']:\n                        append(label,old.fit_standard(XX,yy,X[test],y[test],fit,val,label))\n                    for label in repnames:\n                        append(label,sep.fit_representation(XX,yy,X[test],y[test],fit,val,maps,label))\n                    pd.DataFrame(rows).to_csv(out/\'all_runs.csv\',index=False)\n                    print(f\'{dataset}: sensors={sensors}, seed={seed+1}/{seeds}, labels={budget}; {len(rows)} fits; {time.perf_counter()-start:.1f}s\',flush=True)\n        for dataset,(X,yy) in tasks.items():run_task(dataset,X,yy,\'label-scarcity\',0)\n        shared=train_test_split(np.arange(len(y)),test_size=.25,stratify=y,random_state=SPLIT_SEED)\n        for count in sensor_budgets:\n            idx=sep.SENSORS[count];columns=idx+[i+8 for i in idx]\n            maps=sep.representation_maps(idx,ROOTS);maps.update(arithmetic_maps(idx))\n            run_task(\'fusion_sensor_fresh\',fusion_X[:,columns],y,\'sensor-scarcity\',count,maps,shared)\n    write_tables(rows,out)\n    (out/\'splits.json\').write_text(json.dumps(splits))\n    duration=time.perf_counter()-start\n    (out/\'run_manifest.json\').write_text(json.dumps({\'elapsed_seconds\':duration,\'completed_fits\':len(rows),\n      \'source_hashes\':sources,\'version\':VERSION,\'environment\':environment},indent=2))\n    print(f\'Finished {len(rows)} new fits in {duration:.1f}s: {out.resolve()}\',flush=True)\n    return pd.DataFrame(rows)\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--mode\',choices=[\'quick\',\'full\'],default=\'quick\')\n    parser.add_argument(\'--seeds\',type=int)\n    parser.add_argument(\'--out\',default=\'kernel_family_results\')\n    parser.add_argument(\'--csv\');parser.add_argument(\'--target\')\n    parser.add_argument(\'--drop-columns\',nargs=\'*\',default=[])\n    args=parser.parse_args()\n    if args.seeds is not None and args.seeds<2:parser.error(\'Use at least two seeds\')\n    main(args.out,args.mode,args.seeds,args.csv,args.target,args.drop_columns)\n', 'kernel_geometry_checks.py': '"""Numerical spectral/compression/noise diagnostics; no predictive claims."""\nimport argparse\nimport json\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom sklearn.preprocessing import StandardScaler\nfrom threadpoolctl import threadpool_limits\nimport spectral_scarcity_experiment as old\nimport separation_navigation_experiment as sep\nfrom kernel_family_benchmark import ROOTS,arithmetic_maps\n\n\ndef main(out=\'kernel_geometry_results\'):\n    out=Path(out);out.mkdir(parents=True,exist_ok=True)\n    rng=np.random.default_rng(2026100298);A=StandardScaler().fit_transform(rng.normal(size=(160,16)))\n    maps=sep.representation_maps(list(range(8)),ROOTS);maps.update(arithmetic_maps(list(range(8))))\n    builders={n:(lambda X,Y,n=n:old.kernel(X,Y,n,.05)) for n in old.KERNELS}\n    for (n,mass),M in maps.items():\n        if n.startswith(\'Resolvent\') and mass==.2:\n            builders[n]=lambda X,Y,M=M:(X@M)@(Y@M).T\n    rows=[];residuals=[];perturbations=[]\n    order=rng.permutation(len(A))\n    with threadpool_limits(limits=1):\n        for name,fn in builders.items():\n            K=fn(A,A);e=np.linalg.eigvalsh((K+K.T)/2);scale=max(e.max(),1)\n            assert e.min()/scale>-1e-10\n            v=e[e>scale*1e-10];p=v/v.sum();entropy=float(-np.sum(p*np.log(p)))\n            centered=K-K.mean(axis=0)[None,:]-K.mean(axis=1)[:,None]+K.mean()\n            ce=np.linalg.eigvalsh((centered+centered.T)/2);cv=ce[ce>max(ce.max(),1)*1e-10]\n            rows.append({\'model\':name,\'n\':160,\'d\':16,\'mass\':.2 if name.startswith(\'Resolvent\') else None,\n              \'relative_min_eigenvalue\':float(e.min()/scale),\'numerical_rank\':len(v),\n              \'centered_numerical_rank\':len(cv),\'effective_rank\':float(np.exp(entropy)),\n              \'top10_trace_fraction\':float(v[-10:].sum()/v.sum()),\'seed\':2026100298})\n            for count in [16,32,64]:\n                idx=order[:count];C=K[:,idx];W=K[np.ix_(idx,idx)]\n                approx=C@np.linalg.pinv(W,rcond=1e-10)@C.T\n                residuals.append({\'model\':name,\'landmarks\':count,\n                  \'relative_frobenius_residual\':float(np.linalg.norm(K-approx)/np.linalg.norm(K)),\n                  \'landmark_policy\':\'shared nested random subset\',\'seed\':2026100298})\n            for noise in [1e-3,1e-2,.1]:\n                # Fixed independent perturbation per scale, identical across methods.\n                noisy=A+noise*np.random.default_rng(2026100299).normal(size=A.shape)\n                perturbed=fn(noisy,noisy)\n                perturbations.append({\'model\':name,\'feature_noise_std\':noise,\n                  \'relative_gram_change\':float(np.linalg.norm(perturbed-K)/np.linalg.norm(K))})\n    pd.DataFrame(rows).to_csv(out/\'spectrum.csv\',index=False)\n    pd.DataFrame(residuals).to_csv(out/\'nystrom_residuals.csv\',index=False)\n    pd.DataFrame(perturbations).to_csv(out/\'noise_sensitivity.csv\',index=False)\n    (out/\'protocol.json\').write_text(json.dumps({\'data\':\'160 artificial standardized Gaussian vectors; not fusion trajectories\',\n      \'gamma\':.05,\'mass\':.2,\'seed\':2026100298,\'nystrom\':\'unregularized pseudoinverse at rcond=1e-10\',\n      \'claims\':\'Computational diagnostics only; rank and robustness do not establish prediction or quantum advantage\'},indent=2))\n    print(\'Geometry checks completed:\',out.resolve())\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--out\',default=\'kernel_geometry_results\');a=p.parse_args();main(a.out)\n'}
FAMILY_DIR = OUT/"latest_family"
ENGINE_DIR = OUT/"latest_family_engines"
FAMILY_DIR.mkdir(parents=True,exist_ok=True);ENGINE_DIR.mkdir(parents=True,exist_ok=True)
if CONFIG["RUN_LATEST_FAMILY"]:
    for filename, text in LATEST_ENGINE_SOURCES.items():
        digest=hashlib.sha256(text.encode()).hexdigest()
        assert digest == PROJECT_PROVENANCE["source_engine_sha256"][filename]
        (ENGINE_DIR/filename).write_text(text,encoding="utf-8")
    import importlib
    sys.path.insert(0,str(ENGINE_DIR.resolve()))
    importlib.invalidate_caches()
    # Remove previous imports of these engine names before an in-session rerun.
    for name in [Path(f).stem for f in LATEST_ENGINE_SOURCES]:
        sys.modules.pop(name,None)
    import kernel_family_benchmark as latest_family
    import kernel_geometry_checks as latest_geometry
    import separation_navigation_experiment as latest_separation
    import spectral_scarcity_experiment as latest_scarcity
    (FAMILY_DIR/"source_provenance.json").write_text(json.dumps(PROJECT_PROVENANCE,indent=2))
    exact_checks=latest_family.checks()
    (FAMILY_DIR/"exact_engine_correctness.json").write_text(json.dumps(exact_checks,indent=2))
    # Full classical geometry suite remains in the reference notebook.
    print("Imported latest version:",latest_family.VERSION)
    print("Verified unchanged embedded engines:",len(LATEST_ENGINE_SOURCES))

In [ ]:
family_map_rows=[]
family_kernels={}
family_explicit_features={}
if CONFIG["RUN_LATEST_FAMILY"]:
    from sklearn.preprocessing import StandardScaler
    nr=CONFIG["FAMILY_ARTIFICIAL_ROWS"]
    assert nr>=8
    family_rng=np.random.default_rng(CONFIG["FAMILY_SEED"])
    observed_full=StandardScaler().fit_transform(family_rng.normal(size=(nr,16)))
    np.save(FAMILY_DIR/"artificial_observed_vectors.npy",observed_full)
    for budget,indices in latest_separation.SENSORS.items():
        cols=np.r_[indices,np.asarray(indices)+8]
        X=observed_full[:,cols]
        maps=latest_separation.representation_maps(indices,latest_family.ROOTS)
        maps.update(latest_family.arithmetic_maps(indices))
        for (name,mass),M in maps.items():
            if not name.startswith("Resolvent"):continue
            F=X@M;K=F@F.T
            vals=np.linalg.eigvalsh((K+K.T)/2)
            scale=max(vals[-1],1.0)
            rank=int(np.count_nonzero(vals>scale*1e-10))
            assert vals[0]>=-1e-10*scale and rank<=X.shape[1]
            original_copy=(X.copy()@M)@F.T
            parity=np.linalg.norm(K-original_copy)/np.linalg.norm(K)
            assert parity<1e-12
            family_map_rows.append({"carrier":name,"mass":mass,"observed_times":budget,
              "input_scalar_channels":X.shape[1],"map_output_features":M.shape[1],
              "numerical_rank":rank,"relative_min_eigenvalue":vals[0]/scale,
              "consistent_extension_error":parity})
            if budget==8 and mass==.2:
                family_kernels[name]=K
                family_explicit_features[name]=F
    for name in latest_scarcity.KERNELS:
        K=latest_scarcity.kernel(observed_full,observed_full,name,.05)
        family_kernels[name]=K
        # All labels below are artificial training labels used ONLY for mapping checks.
        training_y=np.tile([0,1],(nr+1)//2)[:nr]
        for mode,strength in [("original",1),("weighted",0),("weighted",1),("sham",1)]:
            weighted,cross,diag=latest_scarcity.spectral_map(K,K,training_y,mode,strength,CONFIG["FAMILY_SEED"])
            assert np.linalg.norm(weighted-cross)/np.linalg.norm(weighted)<1e-8
            label=f"{name} | {mode} strength={strength}"
            family_kernels[label]=weighted
            np.save(FAMILY_DIR/(label.replace(" | ","_").replace(" ","_")+".npy"),weighted)
    # Also verify the exact zero-weight identity for each frozen kernel.
    for name in latest_scarcity.KERNELS:
        assert np.max(np.abs(family_kernels[f"{name} | original strength=1"]-
                      family_kernels[f"{name} | weighted strength=0"]))<1e-12
    family_map_df=pd.DataFrame(family_map_rows)
    family_map_df.to_csv(FAMILY_DIR/"01_all_carrier_mass_sensor_checks.csv",index=False)
    display(family_map_df.groupby("observed_times").agg(
      maps=("carrier","count"),max_rank=("numerical_rank","max"),
      max_input_channels=("input_scalar_channels","max")))
    print("Gram operators retained for circuit checks:",len(family_kernels))

In [ ]:
# Same prime-positive, smooth and randomized measures as the uploaded reference.
CONFIG.update({'ENERGY_MIN':10.,'ENERGY_MAX':50.,'GRID_POINTS':4096,
               'PRIME_POWER_CUTOFF':3000,'PRIME_DAMPING':4.5,'PRIME_COUPLING':.65,
               'MAX_JACOBI_DIM':8})
rng=np.random.default_rng(SEED)
def primes_upto(n):
    sieve = np.ones(n + 1, dtype=bool)
    sieve[:2] = False
    for p in range(2, int(n**0.5) + 1):
        if sieve[p]:
            sieve[p*p:n+1:p] = False
    return np.flatnonzero(sieve)

def prime_power_terms(cutoff, damping):
    ps = primes_upto(cutoff)
    frequencies, amplitudes = [], []
    for p in ps:
        q, k = int(p), 1
        while q <= cutoff:
            t = k * np.log(float(p))
            a = np.log(float(p)) / (float(p)**(0.5*k))
            a *= np.exp(-(t/damping)**2)
            frequencies.append(t)
            amplitudes.append(a)
            if q > cutoff // int(p):
                break
            q *= int(p)
            k += 1
    return np.asarray(frequencies), np.asarray(amplitudes)

def candidate_weight(E, frequencies, amplitudes, coupling,
                     randomized=False, seed=0):
    frequencies = np.asarray(frequencies, float).copy()
    amplitudes = np.asarray(amplitudes, float).copy()
    local_rng = np.random.default_rng(seed)
    phases = np.zeros_like(frequencies)
    if randomized:
        frequencies = local_rng.permutation(frequencies)
        phases = local_rng.uniform(0, 2*np.pi, len(frequencies))
    signal = np.cos(E[:, None]*frequencies[None, :] + phases[None, :]) @ amplitudes
    signal /= max(np.sqrt(0.5*np.sum(amplitudes**2)), 1e-15)
    middle = 0.5*(CONFIG["ENERGY_MIN"] + CONFIG["ENERGY_MAX"])
    sigma = 0.42*(CONFIG["ENERGY_MAX"] - CONFIG["ENERGY_MIN"])
    logw = -0.5*((E-middle)/sigma)**2 + coupling*signal
    logw -= np.max(logw)
    w = np.exp(logw)
    w /= np.sum(w)
    return w, signal

E_GRID = np.linspace(CONFIG["ENERGY_MIN"], CONFIG["ENERGY_MAX"],
                     CONFIG["GRID_POINTS"])
frequencies, amplitudes = prime_power_terms(
    CONFIG["PRIME_POWER_CUTOFF"], CONFIG["PRIME_DAMPING"]
)
w_prime, prime_signal = candidate_weight(
    E_GRID, frequencies, amplitudes, CONFIG["PRIME_COUPLING"]
)
w_random, random_signal = candidate_weight(
    E_GRID, frequencies, amplitudes, CONFIG["PRIME_COUPLING"],
    randomized=True, seed=CONFIG["SEED"] + 1
)
w_smooth, _ = candidate_weight(
    E_GRID, frequencies, amplitudes, 0.0
)

print(f"Prime-power terms: {len(frequencies)}")
print(f"Positive weights: prime={w_prime.min():.3e}, "
      f"random={w_random.min():.3e}, smooth={w_smooth.min():.3e}")
assert min(w_prime.min(), w_random.min(), w_smooth.min()) > 0

def jacobi_from_discrete_measure(nodes, weights, degree):
    nodes = np.asarray(nodes, float)
    weights = np.asarray(weights, float)
    if np.any(weights <= 0):
        raise ValueError("Stieltjes measure must be strictly positive here.")
    q = np.sqrt(weights / weights.sum())
    q /= np.linalg.norm(q)
    q_prev = np.zeros_like(q)
    Q, alpha, beta = [], [], []
    beta_prev = 0.0
    for j in range(degree):
        z = nodes*q
        if j:
            z -= beta_prev*q_prev
        aj = float(q @ z)
        z -= aj*q
        # Two-pass full re-orthogonalization.
        for _ in range(2):
            for qi in Q:
                z -= (qi @ z)*qi
        alpha.append(aj)
        Q.append(q.copy())
        if j < degree - 1:
            bj = float(np.linalg.norm(z))
            if not np.isfinite(bj) or bj <= 100*np.finfo(float).eps:
                raise RuntimeError(f"Lanczos breakdown at degree {j}: b={bj}")
            beta.append(bj)
            q_prev, q = q, z/bj
            beta_prev = bj
    J = np.diag(alpha)
    if beta:
        J += np.diag(beta, 1) + np.diag(beta, -1)
    orth_resid = np.linalg.norm(np.column_stack(Q).T @ np.column_stack(Q)
                                - np.eye(degree))
    return J, np.asarray(alpha), np.asarray(beta), orth_resid

JACOBI = {}
for name, weights in {
    "Prime-positive": w_prime,
    "Smooth-only": w_smooth,
    "Randomized-prime": w_random,
}.items():
    J, a, b, orth = jacobi_from_discrete_measure(
        E_GRID, weights, CONFIG["MAX_JACOBI_DIM"]
    )
    JACOBI[name] = {"Jmax": J, "a": a, "b": b, "orth_resid": orth}

construction_rows = []
for name, obj in JACOBI.items():
    J = obj["Jmax"]
    construction_rows.append({
        "candidate": name,
        "max_dimension": len(J),
        "hermiticity_residual": np.linalg.norm(J-J.T)/np.linalg.norm(J),
        "minimum_b": obj["b"].min(),
        "lanczos_orthogonality_residual": obj["orth_resid"],
        "spectral_min": np.linalg.eigvalsh(J).min(),
        "spectral_max": np.linalg.eigvalsh(J).max(),
    })
construction_df = pd.DataFrame(construction_rows)
display(construction_df)
construction_df.to_csv(OUT / "01_jacobi_construction.csv", index=False)

## Optional exported Gram matrices
Set `INPUT_BUNDLE` to JSON exported by either companion. It contains matrices and provenance, never credentials. It replaces the default family matrices only; prime-Jacobi controls remain separate. Default data are artificial. A supplied real-data bundle does not itself perform a held-out predictive test.

In [ ]:
bundle_provenance={'data':'artificial standardized Gaussian vectors','source':PROJECT_PROVENANCE}
if INPUT_BUNDLE is not None:
    bundle=json.loads(Path(INPUT_BUNDLE).read_text())
    assert bundle['schema']=='project-qpu-gram-v1'
    family_kernels={k:np.asarray(v,dtype=float) for k,v in bundle['grams'].items()}
    assert family_kernels and all(K.ndim==2 and K.shape[0]==K.shape[1] and 4<=len(K)<=32
                                  and np.isfinite(K).all() for K in family_kernels.values())
    bundle_provenance=bundle.get('provenance',{})
    print('Loaded',len(family_kernels),'matrices from supplied bundle.')
(OUT/'input_provenance.json').write_text(json.dumps(bundle_provenance,indent=2))

In [ ]:
from qiskit.circuit.library import QFTGate, StatePreparation

def qpe_circuit(unitary, initial_state, phase_qubits, measure=True):
    size = len(unitary)
    ns = int(np.log2(size))
    if size != 2**ns or ns + phase_qubits > CONFIG["MAX_TOTAL_QUBITS"]:
        raise ValueError("Invalid dimension or simulator size limit exceeded")
    qc = QuantumCircuit(phase_qubits + ns)
    if measure: qc.add_register(ClassicalRegister(phase_qubits, "meas"))
    system = list(range(phase_qubits, phase_qubits + ns))
    state = np.asarray(initial_state, complex)
    if not np.isclose(np.linalg.norm(state), 1):
        raise ValueError("Initial state must be normalized")
    if not np.allclose(state, np.eye(size)[0]):
        qc.append(StatePreparation(state), system)
    qc.h(list(range(phase_qubits)))
    for j in range(phase_qubits):
        power = np.linalg.matrix_power(unitary, 2**j)
        controlled = np.zeros((2*size, 2*size), complex)
        controlled[0::2, 0::2] = np.eye(size)
        controlled[1::2, 1::2] = power
        qc.append(UnitaryGate(controlled, label=f"controlled U^{2**j}"), [j, *system])
    qc.append(QFTGate(phase_qubits).inverse(), list(range(phase_qubits)))
    if measure:
        qc.measure(list(range(phase_qubits)), list(range(phase_qubits)))
    return qc

In [ ]:
def signed_circuit(a,b):
    nq=int(np.log2(len(a)))
    qc=QuantumCircuit(nq+1);qc.add_register(ClassicalRegister(1,'meas'))
    qc.append(StatePreparation(np.r_[a,b]/np.sqrt(2)),range(nq+1))
    qc.h(nq);qc.measure(nq,0)
    return qc

def fidelity_circuit(a,b):
    nq=int(np.log2(len(a)));qc=QuantumCircuit(nq);qc.add_register(ClassicalRegister(nq,'meas'))
    qc.append(StatePreparation(a),range(nq));qc.append(StatePreparation(b).inverse(),range(nq))
    qc.measure(range(nq),range(nq));return qc

def exact_pmf(qc):
    bare=qc.remove_final_measurements(inplace=False)
    measured=[(bare.find_bit(inst.qubits[0]).index,qc.find_bit(inst.clbits[0]).index)
              for inst in qc.data if inst.operation.name=='measure']
    qubits=[q for q,c in sorted(measured,key=lambda x:x[1])]
    return Statevector.from_instruction(bare).probabilities(qubits)

records=[];circuits=[];factor_rows=[]
def add(qc,kind,model,**extra):
    qc.name='experiment_'+str(len(circuits))
    rec={'id':len(circuits),'kind':kind,'model':model,'logical_qubits':qc.num_qubits,**extra}
    rec['exact_pmf']=exact_pmf(qc).tolist()
    records.append(rec);circuits.append(qc)

free_names=['RBF','ProcessUniform','ProcessLamb-PSD','BosonicX-LogPoly-RG']
models=free_names if PROFILE=='free' else list(family_kernels)
missing=set(models)-set(family_kernels)
if missing: raise ValueError('Bundle lacks selected models: '+str(missing)+'. Use full profile or export these kernels.')
for name in models:
    Kfull=family_kernels[name];dimension=8 if len(Kfull)>=8 else 4
    K=np.asarray(Kfull[:dimension,:dimension]);K=(K+K.T)/2
    vals,V=np.linalg.eigh(K);scale=max(float(vals[-1]),1.)
    if vals[0]<-1e-9*scale: raise ValueError('Non-PSD matrix: '+name)
    F=V*np.sqrt(np.maximum(vals,0))[None,:];norms=np.linalg.norm(F,axis=1)
    assert norms.min()>1e-12
    A=F/norms[:,None];res=float(np.linalg.norm(F@F.T-K)/np.linalg.norm(K))
    assert res<1e-8
    factor_rows.append({'model':name,'dimension':dimension,'relative_reconstruction_error':res,
                        'discarded_negative_sum':float(-vals[vals<0].sum())})
    pairs=[(0,1)] if PROFILE=='free' else [(0,1),(0,2),(1,3)]
    for i,j in pairs:
        overlap=float(A[i]@A[j]);extra={'row_i':i,'row_j':j,'exact_overlap':overlap,
          'raw_gram':float(K[i,j]),'norm_product':float(norms[i]*norms[j]),'finite_factor_dimension':dimension}
        add(signed_circuit(A[i],A[j]),'signed',name,**extra)
        assert abs(2*records[-1]['exact_pmf'][0]-1-overlap)<1e-9
        if PROFILE=='full':
            add(fidelity_circuit(A[i],A[j]),'fidelity',name,**extra)
            assert abs(records[-1]['exact_pmf'][0]-overlap**2)<1e-9
    if PROFILE=='full':
        trace=float(np.trace(K));phase=1/8+.75*vals/trace
        unitary=expm(2j*np.pi*(np.eye(dimension)/8+.75*K/trace))
        for case,state in [('largest_eigenstate',V[:,-1]),('basis_zero',np.eye(dimension)[0])]:
            add(qpe_circuit(unitary,state,QPE_PHASE_QUBITS),'gram_qpe',name,
                input_case=case,trace_scale=trace,phase_offset=1/8,phase_scale=.75,
                exact_largest_eigenvalue=float(vals[-1]),phase_qubits=QPE_PHASE_QUBITS)

# Known signed overlaps include -1 to catch accidental squaring or ancilla-bit mistakes.
a=np.array([1.,0.])
for label,b in [('plus_one',a),('zero',np.array([0.,1.])),('minus_one',-a)]:
    add(signed_circuit(a,b),'signed_control',label,exact_overlap=float(a@b),norm_product=1.)
phi=.375
add(qpe_circuit(np.diag([1,np.exp(2j*np.pi*phi)]),np.array([0.,1.]),3),'phase_control','exact_phase_3_over_8',phase_qubits=3)
assert np.argmax(records[-1]['exact_pmf'])==3 and records[-1]['exact_pmf'][3]>1-1e-9
if PROFILE=='full':
    for name,obj in JACOBI.items():
        J=obj['Jmax'];val,V=np.linalg.eigh(J)
        center=float((val[0]+val[-1])/2);radius=float((val[-1]-val[0])/2*1.25)
        H=.5*np.eye(len(J))+(J-center*np.eye(len(J)))/(2*radius)
        for case,state in [('largest_eigenstate',V[:,-1]),('basis_zero',np.eye(len(J))[0])]:
            add(qpe_circuit(expm(2j*np.pi*H),state,QPE_PHASE_QUBITS),'jacobi_qpe',name,
                input_case=case,energy_center=center,energy_radius=radius,
                exact_largest_eigenvalue=float(val[-1]),phase_qubits=QPE_PHASE_QUBITS)
pd.DataFrame(factor_rows).to_csv(OUT/'factorization.csv',index=False)
(OUT/'experiments.json').write_text(json.dumps(records,indent=2))
with (OUT/'logical_circuits.qpy').open('wb') as f:qpy.dump(circuits,f)
print('Constructed',len(circuits),'circuits. All ideal mapping checks passed.')

## Connect and compile
Only a new API-key connection with `plans_preference=['open']` is used; saved paid accounts and qBraid paid QPU gateways are not used. An operational backend must be available to your Open Plan. Local mode uses a 20-qubit fake IBM backend **only for compilation**; measurements below are ideal Aer shots. Logical register `meas` survives physical-layout changes.

Depth and gate guards are conservative heuristics, not guarantees of device fidelity or runtime. Full mode exposes every circuit even if a guard blocks it. Investigate the report before increasing limits.

In [ ]:
service=None
if RUN_HARDWARE:
    api_key=os.environ.get('IBM_QUANTUM_API_KEY') or getpass.getpass('IBM Quantum API key (hidden): ')
    service=QiskitRuntimeService(channel='ibm_quantum_platform',token=api_key,
                                plans_preference=['open'],region='us-east')
    del api_key
    backend=service.backend(BACKEND_NAME) if BACKEND_NAME else service.least_busy(operational=True,simulator=False)
    assert not backend.configuration().simulator
else:
    from qiskit_ibm_runtime.fake_provider import FakeAlmadenV2
    backend=FakeAlmadenV2()

start=CHUNK_INDEX*CHUNK_SIZE
selected_ids=list(range(start,min(start+CHUNK_SIZE,len(circuits))))
if not selected_ids:raise ValueError('Chunk index is beyond the suite.')
selected=[circuits[i] for i in selected_ids]
compiled=transpile(selected,backend=backend,optimization_level=2,seed_transpiler=SEED,num_processes=1)
cost=[]
for i,qc in zip(selected_ids,compiled):
    two=sum(inst.operation.num_qubits==2 and inst.operation.name not in ['barrier'] for inst in qc.data)
    depth=qc.depth(filter_function=lambda inst:inst.operation.num_qubits==2 and inst.operation.name!='barrier')
    ok=two<=MAX_TWO_QUBIT_GATES and depth<=MAX_TWO_QUBIT_DEPTH and circuits[i].num_qubits<=MAX_LOGICAL_QUBITS
    cost.append({'id':i,'model':records[i]['model'],'kind':records[i]['kind'],
                 'logical_qubits':circuits[i].num_qubits,'physical_register_size':qc.num_qubits,
                 'total_depth':qc.depth(),'two_qubit_gates':two,'two_qubit_depth':depth,'within_guards':ok})
COST=pd.DataFrame(cost);display(COST)
COST.to_csv(OUT/f'compile_chunk_{CHUNK_INDEX}.csv',index=False)
with (OUT/f'compiled_chunk_{CHUNK_INDEX}.qpy').open('wb') as f:qpy.dump(compiled,f)
print('Backend:',backend.name,'Chunk:',CHUNK_INDEX,'Shots:',SHOTS,'Requested samples:',len(selected)*SHOTS)

## Submit one chunk, with resume protection
Local mode runs the selected logical circuits on Aer. Hardware mode reserves the **maximum job timeout** against the notebook budget before submission, so unknown/queued jobs remain reserved. This ledger only covers jobs in this output folder: check your dashboard for other jobs and rolling-window usage. Runtime timeouts can terminate an incomplete experiment.

Repeated execution retrieves the saved job instead of resubmitting it. If submission fails after the server may have accepted it, an `unknown_submission` record blocks retries: inspect IBM Workloads and recover the job ID manually. Never delete a live-job ledger to retry. Job mode is used without sessions.

In [ ]:
LEDGER_PATH=OUT/'submission_ledger.json'
ledger=json.loads(LEDGER_PATH.read_text()) if LEDGER_PATH.exists() else []
fingerprint=hashlib.sha256((OUT/f'compiled_chunk_{CHUNK_INDEX}.qpy').read_bytes()+
                          json.dumps({'shots':SHOTS,'backend':backend.name,'ids':selected_ids},sort_keys=True).encode()).hexdigest()
job=None;result=None;current_entry=None
if not RUN_HARDWARE:
    aer=AerSimulator(method="statevector",seed_simulator=SEED,max_parallel_threads=1,max_parallel_experiments=1)
    local_compiled=transpile(selected,basis_gates=["u","cx"],optimization_level=1,seed_transpiler=SEED,num_processes=1)
    local_counts=aer.run(local_compiled,shots=SHOTS).result().get_counts()
    if isinstance(local_counts,dict):local_counts=[local_counts]
else:
    existing=[e for e in ledger if e['fingerprint']==fingerprint]
    if existing:
        current_entry=existing[-1]
        if not current_entry.get('job_id'):
            raise RuntimeError('Unknown submission. Recover job ID from IBM Workloads before retrying.')
        job=service.job(current_entry['job_id'])
        print('Resuming existing job:',job.job_id())
    else:
        if not COST.within_guards.all():raise RuntimeError('Compilation guards failed. No QPU job submitted.')
        reserved=sum(e['reserved_seconds'] for e in ledger)
        if reserved+MAX_JOB_SECONDS>min(RUN_BUDGET_SECONDS,AVAILABLE_QPU_SECONDS):
            raise RuntimeError('Reserved job time exceeds remaining configured allowance. No job submitted.')
        current_entry={'fingerprint':fingerprint,'ids':selected_ids,'shots':SHOTS,'backend':backend.name,
                       'reserved_seconds':MAX_JOB_SECONDS,'job_id':None,'status':'unknown_submission'}
        ledger.append(current_entry)
        LEDGER_PATH.write_text(json.dumps(ledger,indent=2))
        sampler=Sampler(mode=backend)
        sampler.options.max_execution_time=MAX_JOB_SECONDS
        sampler.options.twirling.enable_gates=False
        sampler.options.twirling.enable_measure=False
        sampler.options.dynamical_decoupling.enable=False
        job=sampler.run(compiled,shots=SHOTS)
        current_entry.update(job_id=job.job_id(),status=str(job.status()))
        LEDGER_PATH.write_text(json.dumps(ledger,indent=2))
        print('Submitted job:',job.job_id(),'. Saved immediately; rerun retrieval when complete.')

## Retrieve and analyse (rerun this cell for a queued job)
No long queue wait occurs. Confidence intervals describe binomial shot uncertainty only; hardware bias is additional. The signed estimate is `2P(ancilla=0)-1`; the fidelity estimate is the all-zero probability. Raw Gram recovery multiplies signed overlap by the two row norms. QPE comparisons use the full finite-register distribution; a mixed-state peak is not automatically assigned to a single eigenvalue.

In [ ]:
def wilson(success,total,z=1.959963984540054):
    p=success/total;den=1+z*z/total
    center=(p+z*z/(2*total))/den
    half=z*np.sqrt(p*(1-p)/total+z*z/(4*total*total))/den
    return max(0.,center-half),min(1.,center+half)

counts_list=None
if RUN_HARDWARE:
    status=str(job.status()).upper();print('Job status:',status)
    current_entry['status']=status;LEDGER_PATH.write_text(json.dumps(ledger,indent=2))
    if 'DONE' in status:
        result=job.result()
        counts_list=[pub.data.meas.get_counts() for pub in result]
        (OUT/f'job_metrics_chunk_{CHUNK_INDEX}.json').write_text(json.dumps(job.metrics(),indent=2,default=str))
        (OUT/f'pub_metadata_chunk_{CHUNK_INDEX}.json').write_text(json.dumps([pub.metadata for pub in result],indent=2,default=str))
    elif any(x in status for x in ['ERROR','CANCELLED']):
        raise RuntimeError('QPU job ended without a complete result. Inspect IBM Workloads; no automatic resubmission.')
    else:print('Still queued/running. Rerun this cell later; do not resubmit.')
else:counts_list=local_counts

if counts_list is not None:
    rows=[]
    for i,counts in zip(selected_ids,counts_list):
        rec=records[i];m=circuits[i].num_clbits;pmf=np.zeros(2**m)
        for bit,count in counts.items():pmf[int(bit.replace(' ',''),2)]+=count
        total=int(pmf.sum());assert total==SHOTS
        p0=pmf[0]/total;pmf/=total
        row={'id':i,'model':rec['model'],'kind':rec['kind'],'shots':total,
             'hardware_run':RUN_HARDWARE,'backend':backend.name if RUN_HARDWARE else 'Aer ideal',
             'distribution_TV':float(.5*np.abs(pmf-np.array(rec['exact_pmf'])).sum())}
        lo,hi=wilson(counts.get('0'*m,0),total)
        if rec['kind'] in ['signed','signed_control']:
            estimate=2*p0-1
            row.update(estimate=estimate,exact=rec['exact_overlap'],ci95_low=2*lo-1,ci95_high=2*hi-1,
                       absolute_error=abs(estimate-rec['exact_overlap']),
                       raw_gram_estimate=estimate*rec['norm_product'])
        elif rec['kind']=='fidelity':
            row.update(estimate=p0,exact=rec['exact_overlap']**2,ci95_low=lo,ci95_high=hi,
                       absolute_error=abs(p0-rec['exact_overlap']**2))
        else:
            phase=int(pmf.argmax())/2**m;row.update(modal_phase=phase,input_case=rec.get('input_case','calibration'))
            if rec['kind']=='gram_qpe':row['modal_decoded_eigenvalue']=rec['trace_scale']*(phase-1/8)/.75
            if rec['kind']=='jacobi_qpe':row['modal_decoded_energy']=rec['energy_center']+rec['energy_radius']*(2*phase-1)
        rows.append(row)
    pd.DataFrame(rows).to_csv(OUT/f'results_chunk_{CHUNK_INDEX}.csv',index=False)
    (OUT/f'counts_chunk_{CHUNK_INDEX}.json').write_text(json.dumps(counts_list,indent=2))
    display(pd.DataFrame(rows))

In [ ]:
manifest={'date':'2026-10-03','profile':PROFILE,'versions':ENV,'seed':SEED,
          'hardware_requested':RUN_HARDWARE,'hardware_results_retrieved':RUN_HARDWARE and counts_list is not None,
          'quantum_advantage_established':False,'classical_factorization':True,
          'normalization_changes_raw_kernel':True,'chunk_index':CHUNK_INDEX,
          'shots_per_circuit':SHOTS,'full_suite_circuits':len(circuits),
          'selected_ids':selected_ids,'run_budget_seconds':RUN_BUDGET_SECONDS,
          'job_timeout_seconds':MAX_JOB_SECONDS,'backend':backend.name,
          'source_provenance':PROJECT_PROVENANCE}
(OUT/f'manifest_chunk_{CHUNK_INDEX}.json').write_text(json.dumps(manifest,indent=2))
(OUT/'environment_freeze.txt').write_text('\n'.join(f'{p}=={v}' for p,v in ENV.items())+'\n')
archive=shutil.make_archive(str(OUT),'zip',root_dir=OUT)
print('Download results:',archive)

## Current official references
- [qBraid pricing](https://qbraid.com/pricing): free CPU/notebook access; its paid hardware gateways use credits.
- [IBM Open Plan](https://quantum.cloud.ibm.com/docs/en/guides/plans-overview): 10 QPU minutes per rolling 28-day window; check your account and any promotion.
- [qBraid Qiskit environment](https://quantum.cloud.ibm.com/docs/en/guides/online-lab-environments).
- [Open-only instance selection](https://quantum.cloud.ibm.com/docs/en/guides/initialize-account).
- [Executor Sampler](https://quantum.cloud.ibm.com/docs/en/api/qiskit-ibm-runtime/executor-sampler-sampler): current 0.50 client API.
- [QPU timeout](https://quantum.cloud.ibm.com/docs/en/guides/max-execution-time).

A queue delay is not QPU execution time. A small compiled circuit is not a guarantee of high fidelity. The full suite must be spread across allowances if budget guards stop it. Export your ledger and results before closing the cloud session. Credentials are held in memory and are excluded from outputs.